# Oracle Part-Aware Colour Denoising — does semantic part structure make colour more predictable?

A direct extension of `Geometry_Color_Mutual_Denoising_kaggle.ipynb`. Everything that made that
study a controlled comparison is reused **verbatim**: dataset loading, unit-sphere normalisation,
the object-level train/val split, the DDPM forward process, the frozen validation noise bank, the
paired batch/timestep/noise seeding, the tiny-set overfit gate, the full colour metric suite, the
plotting conventions, the checkpoint format and the automatic report generator.

**The hypothesis under test**

> Colour is more predictable when geometry→colour information is exchanged **within the same
> semantic object part**.

The previous notebook established the left half of the ladder — whether clean geometry helps colour
denoising at all. This notebook adds the right half: whether *restricting* that geometric reasoning
to same-part neighbourhoods helps further, and if so, whether the gain comes from merely *knowing*
the part or specifically from *confining information flow* to it.

### The model ladder — each rung adds exactly one thing

| model | colour input | geometry | part information | neighbourhood |
|---|---|---|---|---|
| **C** | $C_t$ | — | — | none (pointwise) |
| **C+Gpw** | $C_t$ | $G_0$ concat | — | none (pointwise) |
| **C+Gloc** | $C_t$ | $G_0$ concat | — | kNN(XYZ), **unmasked** |
| **C+GPid** | $C_t$ | $G_0$ concat | part embedding | kNN(XYZ), **unmasked** |
| **C+GP** | $C_t$ | $G_0$ concat | part embedding | kNN(XYZ), **same-part masked** |

`C` and `C+Gpw` are byte-for-byte the same architectures as the previous notebook's `C` and `C+G`,
so this run is directly comparable to it. `C+Gloc` exists because without it the `C+G → C+GP`
comparison would confound *"added local aggregation"* with *"added part masking"* — and separating
those is the entire point.

### The three questions, kept separate

- **Question A** — does clean geometry help colour denoising? `C` vs `C+Gpw` (and vs `C+Gloc`).
- **Question B** — does semantic part structure add anything beyond geometry? `C+Gloc` vs `C+GP`.
- **Question C** — is any gain from *knowing* the part, or from *restricting flow* to it?
  `C+GPid` vs `C+GP`.

These are never merged into one verdict.

### Design rules carried over, plus one new one

1. **Colour is an attribute, never a coordinate.** Every neighbourhood is built from XYZ alone.
2. **Part labels mask or condition information flow — they never redefine distance.** kNN candidates
   come from XYZ; the part label only decides which of those candidates may contribute. §12 asserts
   this programmatically.
3. **Paired everything.** Same objects, same timesteps, same noise tensors, same kNN topology.
4. **Oracle part labels.** These are ground-truth ShapeNet-Part labels. This measures the
   **upper bound** on what perfect part knowledge could buy — it is not a test-time method.
5. **Geometry stays clean.** $G_0$ is never noised here. The question is narrowly: *given correct
   geometry, does semantic structure make colour easier?*

Before any model is trained, §8 and §9 test whether the premise is even true in this data — whether
colour really is more coherent within parts, and whether part boundaries really are colour
boundaries. If those fail, the architecture is unmotivated and the notebook says so.

## 1 · Configuration

Deliberately identical to the previous notebook wherever it does not have to change: same `SEED`,
`NUM_POINTS`, `SUBSAMPLE`, `T`, `BETA_SCHEDULE`, `LEARNING_RATE`, `NUM_EPOCHS`, `WIDTH`,
`N_BLOCKS`, `BATCH_SIZE`, `EVAL_TIMESTEPS`, `EVAL_REPEATS`, `VAL_BANK_REPEATS`, `KNN_K`,
`N_REGIONS`. The geometry-denoising pair (`G`, `G+C`) is **not** trained here — this notebook is
about colour only — which buys the budget for the five-model colour ladder.

**New knobs** are grouped at the bottom under *part-aware*.

**OOM order:** `NUM_POINTS` (1024→512) → `BATCH_SIZE` (8→4) → `PART_K` (16→8) → `WIDTH` (128→64).

**Runtime.** The three models with kNN aggregation cost roughly 6× the pointwise ones per step, so
the ladder is far from free. `VAL_EVERY` (default 5) is the cheapest lever: the frozen-bank
validation pass is about 40 % of a local model's epoch cost, and sampling it every 5 epochs only
coarsens the curve — the final number is unchanged, because the last epoch is always evaluated.
§16 prints a runtime estimate derived from your own §15 timings *before* training starts.

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

# ─────────── identical to the previous notebook ───────────
SEED             = 0
CATEGORY         = "airplane"   # "airplane" | "car" | "chair"
NUM_POINTS       = 1024
NUM_TRAIN_SHAPES = 150
NUM_VAL_SHAPES   = 40
BATCH_SIZE       = 8
NUM_EPOCHS       = 600
LEARNING_RATE    = 1e-3
WEIGHT_DECAY     = 1e-4
LR_SCHEDULE      = "cosine"
T                = 1000
BETA_SCHEDULE    = "cosine"
WIDTH            = 128
N_BLOCKS         = 3
SUBSAMPLE        = "random"

RUN_SANITY_CHECK    = True
SANITY_CLOUDS       = 6
SANITY_STEPS        = 1500
RUN_FULL_EXPERIMENT = False     # <<< flip to True only after §14 prints "SANITY CHECK PASSED"
RESUME              = True
FORCE_RETRAIN       = False
TIME_BUDGET_MIN_PER_MODEL = None
DETERMINISTIC_STRICT = True

VAL_EVERY        = 5      # compute the frozen-bank val loss every N epochs (and on the last).
                          # The per-epoch val pass is ~40% of a LOCAL model's epoch cost, so this
                          # is the cheapest big saving; it only coarsens the curve, never the
                          # final number. Set to 1 to restore per-epoch validation.

EVAL_TIMESTEPS   = [50, 100, 250, 500, 750, 900]
EVAL_REPEATS     = 4
VAL_BANK_REPEATS = 8
KNN_K            = 8      # metric neighbourhoods (XYZ only)
N_REGIONS        = 32

N_VIS_SHAPES = 3
VIS_T        = 250
VIEW_ELEV, VIEW_AZIM = 22, 135
PT_SIZE      = 3.0
SAVE_PLOTLY  = True

HF_DATASET      = "eylulpelinkilic/Colored_Point_Clouds"
DATA_SOURCE     = "auto"
LOCAL_DATA_ROOT = os.environ.get("PCC_DATA_ROOT", "")
CLONE_REPO = False
REPO_URL    = "https://github.com/eylulpelinkilic/Colored_Point_Cloud_Completion.git"
REPO_BRANCH = "Pelin"

# ─────────── NEW: part-aware settings ───────────
PART_K        = 16     # kNN size for the model's local aggregation (XYZ only, candidates)
PART_EMB      = 8      # learnable part-embedding width for C+GPid / C+GP
MAX_PARTS     = 8      # embedding table size; labels are 0..3 in this dataset
TRAIN_MODELS  = ["C", "C+Gpw", "C+Gloc", "C+GPid", "C+GP"]
# The three LOCAL models dominate the runtime (~6x the pointwise ones). With VAL_EVERY=5 the full
# ladder is ~70 min on a T4 — §16 prints an estimate from YOUR §15 timings before you commit.
# If you must trim: "C", "C+Gloc", "C+GP" is the minimum set that answers Question B; Question C
# additionally needs "C+GPid". Dropping "C+Gpw" saves only ~3 min (it is pointwise).
BOUNDARY_K    = 8      # kNN used to DEFINE semantic boundary points for the metrics
COHERENCE_MIN_PART_PTS = 8   # parts smaller than this are skipped in the coherence statistics

ON_KAGGLE   = os.path.isdir("/kaggle/working")
RESULTS_DIR = "/kaggle/working/results" if ON_KAGGLE else os.path.abspath("./results_part")
CACHE_DIR   = "/kaggle/temp/pcc_cache"  if ON_KAGGLE else os.path.abspath("./.pcc_cache")
REPO_DIR    = "/kaggle/temp/pcc_repo"   if ON_KAGGLE else os.path.abspath("./.pcc_repo")
CKPT_DIR = os.path.join(RESULTS_DIR, "checkpoints")
HIST_DIR = os.path.join(RESULTS_DIR, "history")
TAB_DIR  = os.path.join(RESULTS_DIR, "tables")
FIG_DIR  = os.path.join(RESULTS_DIR, "figures")
for _d in (RESULTS_DIR, CKPT_DIR, HIST_DIR, TAB_DIR, FIG_DIR):
    os.makedirs(_d, exist_ok=True)
try:
    os.makedirs(CACHE_DIR, exist_ok=True)
except OSError:
    CACHE_DIR = os.path.abspath("./.pcc_cache"); os.makedirs(CACHE_DIR, exist_ok=True)

try:
    import torch as _t
    DEVICE = "cuda" if _t.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"

SANITY_PASSED = None

print("ON_KAGGLE   :", ON_KAGGLE)
print("DEVICE      :", DEVICE)
print("RESULTS_DIR :", RESULTS_DIR)
print("category    :", CATEGORY, "| points", NUM_POINTS,
      "| shapes", NUM_TRAIN_SHAPES, "train /", NUM_VAL_SHAPES, "val")
print("models      :", TRAIN_MODELS)
print("RUN_FULL_EXPERIMENT =", RUN_FULL_EXPERIMENT,
      "" if RUN_FULL_EXPERIMENT else "  <- §16 will be skipped until you flip this")

## 2 · Environment / dependencies

Kaggle already ships torch, numpy, pandas, scipy and matplotlib. Only `huggingface_hub` (and
optionally `plotly`) may be missing, so we install just those, quietly, and only if the import fails.

**Inspect:** the CUDA line. If it says `cuda: False` the notebook still runs, but §14–17 will be slow —
turn the accelerator on in *Settings → Accelerator*.

In [ ]:
import importlib, subprocess, sys

def _ensure(module, pip_name=None):
    try:
        importlib.import_module(module); return "present"
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or module], check=False)
        try:
            importlib.import_module(module); return "installed"
        except ImportError:
            return "MISSING"

for _m in ("numpy", "torch", "pandas", "scipy", "matplotlib", "huggingface_hub", "plotly"):
    print(f"  {_m:16s} {_ensure(_m)}")

import numpy as np, torch, pandas as pd, matplotlib, scipy
print()
print("numpy", np.__version__, "| torch", torch.__version__,
      "| pandas", pd.__version__, "| matplotlib", matplotlib.__version__, "| scipy", scipy.__version__)
print("cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu only")
if torch.cuda.is_available():
    print("gpu memory: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
import shutil as _sh
print("free disk : %.0f GB" % (_sh.disk_usage(RESULTS_DIR).free / 1e9))

## 3 · Repository and dataset setup

Reused verbatim from the previous notebook. The loader already returns the per-point `part` array
alongside `xyz` and `rgb` — the previous notebook simply discarded it. Here we keep it.

Source: the **public** HuggingFace dataset `eylulpelinkilic/Colored_Point_Clouds`, path
`labeled_s3/<synset>/<model>.npz`, keys `xyz (8192,3) float32`, `rgb (8192,3) float32 in [0,1]`,
`part (8192,) int16`. No token required.

In [ ]:
import glob, numpy as np

SYNSETS = {"airplane": "02691156", "car": "02958343", "chair": "03001627"}
assert CATEGORY in SYNSETS, f"CATEGORY must be one of {list(SYNSETS)}"
SYNSET = SYNSETS[CATEGORY]

if CLONE_REPO and not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=False)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-q", REPO_BRANCH], check=False)
    if os.path.isdir(REPO_DIR):
        sys.path.insert(0, REPO_DIR); print("repo cloned ->", REPO_DIR)

IS_SYNTHETIC = False
DATA_ORIGIN  = None
FILES        = []


def _local_files():
    if not LOCAL_DATA_ROOT:
        return []
    return sorted(glob.glob(os.path.join(LOCAL_DATA_ROOT, "labeled_s3", SYNSET, "*.npz")))


def _hf_files():
    from huggingface_hub import snapshot_download
    root = snapshot_download(HF_DATASET, repo_type="dataset",
                             allow_patterns=[f"labeled_s3/{SYNSET}/*.npz"],
                             cache_dir=CACHE_DIR)
    return sorted(glob.glob(os.path.join(root, "labeled_s3", SYNSET, "*.npz")))


# --- synthetic fallback -------------------------------------------------------------------
# Adapted from scripts/toy_part_color.py (build_toy / box_surface). Chair-like axis-aligned boxes.
# Each region carries TWO flat materials, mirroring the real finding that a semantic part usually
# holds several flat materials rather than a smooth gradient -> a genuine geometry<->colour relation
# for a smoke test, with a fixed cross-object palette so the relation is learnable.
_TOY_PARTS = [((0.0, 0.90, 0.00), (2.0, 0.15, 2.0)), ((0.0, 1.65, -0.93), (2.0, 1.5, 0.15)),
              ((-0.9, 0.45, -0.9), (0.16, 0.9, 0.16)), ((0.9, 0.45, -0.9), (0.16, 0.9, 0.16)),
              ((-0.9, 0.45, 0.9), (0.16, 0.9, 0.16)), ((0.9, 0.45, 0.9), (0.16, 0.9, 0.16))]
_TOY_PALETTE = np.array([[0.80, 0.22, 0.18], [0.18, 0.35, 0.75], [0.30, 0.30, 0.32],
                         [0.30, 0.30, 0.32], [0.30, 0.30, 0.32], [0.30, 0.30, 0.32],
                         [0.95, 0.88, 0.70], [0.10, 0.55, 0.45], [0.85, 0.85, 0.88],
                         [0.85, 0.85, 0.88], [0.85, 0.85, 0.88], [0.85, 0.85, 0.88]])


def _box_surface(center, size, n, rng):
    center = np.asarray(center, float); hx, hy, hz = np.asarray(size, float) / 2.0
    areas = np.array([hy*hz, hy*hz, hx*hz, hx*hz, hx*hy, hx*hy]) * 4
    counts = rng.multinomial(n, areas / areas.sum()); pts = []
    for face, k in enumerate(counts):
        if k == 0: continue
        u, v = rng.uniform(-1, 1, k), rng.uniform(-1, 1, k); o = np.ones(k)
        p = [np.stack([o, u, v], 1), np.stack([-o, u, v], 1), np.stack([u, o, v], 1),
             np.stack([u, -o, v], 1), np.stack([u, v, o], 1), np.stack([u, v, -o], 1)][face]
        pts.append(p * np.array([hx, hy, hz]) + center)
    return np.concatenate(pts, 0)


def _synthetic_shape(seed, n=8192):
    rng = np.random.default_rng(seed)
    sizes = np.array([s for _, s in _TOY_PARTS], float)
    area = 2 * (sizes[:, 0]*sizes[:, 1] + sizes[:, 1]*sizes[:, 2] + sizes[:, 0]*sizes[:, 2])
    counts = rng.multinomial(n, area / area.sum())
    xyz, rgb, part = [], [], []
    for pid, ((c, s), k) in enumerate(zip(_TOY_PARTS, counts)):
        if k == 0: continue
        c = np.asarray(c, float) * (1 + rng.uniform(-0.10, 0.10, 3))
        s = np.asarray(s, float) * (1 + rng.uniform(-0.10, 0.10, 3))
        p = _box_surface(c, s, k, rng)
        second = p[:, 1] > (c[1])                     # second material on the upper half
        col = np.where(second[:, None], _TOY_PALETTE[pid + 6], _TOY_PALETTE[pid])
        col = col + rng.normal(0, 0.02, (k, 3))
        xyz.append(p); rgb.append(np.clip(col, 0, 1)); part.append(np.full(k, pid))
    return (np.concatenate(xyz).astype(np.float32), np.concatenate(rgb).astype(np.float32),
            np.concatenate(part).astype(np.int16))


_need = NUM_TRAIN_SHAPES + NUM_VAL_SHAPES
_sources = {"auto": ["local", "hf", "synthetic"], "local": ["local"],
            "hf": ["hf"], "synthetic": ["synthetic"]}[DATA_SOURCE]

for src in _sources:
    try:
        if src == "local":
            FILES = _local_files()
        elif src == "hf":
            print("downloading from HuggingFace (needs Internet = ON) ...", flush=True)
            FILES = _hf_files()
        else:
            FILES = [("synthetic", i) for i in range(_need)]
            IS_SYNTHETIC = True
        if len(FILES) >= _need:
            DATA_ORIGIN = src; break
        print(f"  [{src}] only {len(FILES)} shapes found (need {_need}) -> next source")
        FILES = []
    except Exception as e:
        print(f"  [{src}] failed: {type(e).__name__}: {str(e)[:180]}")
        FILES = []

assert DATA_ORIGIN is not None, (
    f"No data source produced {_need} shapes. Turn Kaggle Internet ON, or set "
    f"DATA_SOURCE='synthetic' for a smoke test, or point LOCAL_DATA_ROOT at a PCC_DATA_ROOT.")
assert len(FILES) >= _need

print(f"\nsource   : {DATA_ORIGIN}")
print(f"category : {CATEGORY} ({SYNSET})")
print(f"shapes   : {len(FILES)} available, {_need} will be used")
if IS_SYNTHETIC:
    print("\n" + "!" * 78)
    print("!! SYNTHETIC FALLBACK IS ACTIVE — this is a SMOKE TEST, not a scientific result.")
    print("!! Any conclusion drawn from this run is about toy boxes, not ShapeNet objects.")
    print("!" * 78)

## 4 · Imports and reproducibility

Seeds for `random`, `numpy` and `torch`. `torch.use_deterministic_algorithms(warn_only=True)` asks
CUDA for deterministic kernels but never hard-fails when one is unavailable — **perfect GPU
determinism is not guaranteed**, and the notebook does not claim it. What *is* guaranteed, and is what
the fairness argument actually rests on, is that the paired models receive byte-identical data,
timesteps and noise (§13, §19), because those are drawn from explicitly seeded generators rather than
from ambient RNG state.

In [ ]:
import math, json, time, random, warnings
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers the 3-D projection)
from scipy.spatial import cKDTree
from scipy.spatial.distance import cdist

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 140, "font.size": 9,
                     "axes.grid": True, "grid.alpha": 0.25})


def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


set_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
if DETERMINISTIC_STRICT:
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception as e:
        print("deterministic algorithms not fully available:", type(e).__name__)

DEV = torch.device(DEVICE)
print("seeded with", SEED, "| device", DEV)
print("note: CUDA determinism is best-effort; paired fairness comes from explicit generators, not "
      "from global RNG state.")

## 5 · Dataset inspection — including the part labels

**Inspect:** the part-label block. These are ShapeNet-Part semantic labels, already remapped to
**contiguous, category-local ids 0…P−1** in `labeled_s3` (verified below — no remapping is applied
by this notebook, so no mapping table is needed). For airplane the four parts are
body / wing / tail / engine; chair is back / seat / leg / arm; car is roof / hood / wheel / body.

Watch the *smallest part fraction*: some objects have a part occupying well under 1 % of points,
which after subsampling can leave a handful of points or none. §12's masking has to survive that.

In [ ]:
def load_raw(entry):
    if IS_SYNTHETIC:
        return _synthetic_shape(seed=1000 + entry[1])
    z = np.load(entry)
    return (z["xyz"].astype(np.float32),
            np.clip(z["rgb"].astype(np.float32), 0.0, 1.0),
            z["part"].astype(np.int64))


PART_NAMES = {"airplane": ["body", "wing", "tail", "engine"],
              "chair":    ["back", "seat", "leg", "arm"],
              "car":      ["roof", "hood", "wheel", "body"]}.get(CATEGORY, None)


def part_name(p):
    '''Human-readable part name, safe when a source has more parts than the category name list
    (the synthetic fallback has 6 box parts, for instance).'''
    return PART_NAMES[p] if (PART_NAMES and 0 <= p < len(PART_NAMES)) else f"part{p}"

_xyz, _rgb, _part = load_raw(FILES[0])
name0 = "synthetic-0" if IS_SYNTHETIC else os.path.basename(FILES[0])
print("example shape:", name0)
for nm, a in (("xyz", _xyz), ("rgb", _rgb), ("part", _part)):
    print(f"  {nm:5s} {str(a.shape):12s} {str(a.dtype):9s} min {a.min():8.4f}  max {a.max():8.4f}")
assert len(_xyz) == len(_rgb) == len(_part), "xyz / rgb / part must be index-aligned"

_u, _c = np.unique(_part, return_counts=True)
print(f"\n  parts on this object: {_u.tolist()}")
for p, n in zip(_u, _c):
    nm = part_name(p)
    print(f"    {int(p)} {nm:8s} {n:6d} pts ({100*n/len(_part):5.2f}%)  mean RGB "
          f"{np.round(_rgb[_part == p].mean(0), 3)}")

# ---- corpus-level part statistics ----
from collections import Counter
_np_hist, _smallest, _ids = Counter(), [], set()
for e in FILES[:min(60, len(FILES))]:
    p = load_raw(e)[2]
    u, c = np.unique(p, return_counts=True)
    _np_hist[len(u)] += 1; _smallest.append((c / c.sum()).min()); _ids |= set(u.tolist())
print(f"\nover {sum(_np_hist.values())} objects:")
print(f"  parts per object : {dict(sorted(_np_hist.items()))}")
print(f"  label ids seen   : {sorted(_ids)}  -> contiguous from 0: "
      f"{sorted(_ids) == list(range(len(_ids)))}  (no remapping applied)")
print(f"  smallest part    : median {np.median(_smallest)*100:.2f}% of points, "
      f"min {np.min(_smallest)*100:.3f}%")
assert max(_ids) < MAX_PARTS, f"MAX_PARTS={MAX_PARTS} is too small for labels {sorted(_ids)}"
N_PARTS_MAX = int(max(_ids)) + 1
print(f"  -> N_PARTS_MAX = {N_PARTS_MAX}, embedding table sized {MAX_PARTS}")

## 6 · Preprocessing — carrying the part label through

Identical to the previous notebook, with one change: `subsample` now **returns the index array** so
that `xyz`, `rgb` *and* `part` are gathered with the **same indices**. The random draw is the same
call with the same seed, so the resulting `xyz`/`rgb` are byte-identical to the previous run —
the split, the sampling and the normalisation are unchanged, and only `part` is added.

Correspondence $(G_{0,i}, C_{0,i}, s_i)$ is asserted explicitly after subsampling, not assumed.

In [ ]:
# --- from scripts/run_benchmark_kaggle.py: make_entry() -------------------------------------
def normalize_xyz(xyz):
    c = xyz - xyz.mean(0)
    return (c / (np.linalg.norm(c, axis=1).max() + 1e-9)).astype(np.float32)


def fps_idx(xyz, n, start=0):
    '''Farthest-point sampling, deterministic (fixed start index).'''
    pts = torch.from_numpy(np.ascontiguousarray(xyz)).float()
    N = pts.shape[0]
    idx = torch.zeros(n, dtype=torch.long)
    dist = torch.full((N,), 1e10)
    far = torch.tensor(start)
    for i in range(n):
        idx[i] = far
        dist = torch.minimum(dist, ((pts - pts[far]) ** 2).sum(-1))
        far = torch.max(dist, 0).indices
    return idx.numpy()


def subsample_idx(n_src, xyz, n, seed):
    '''Returns the INDEX array, so xyz / rgb / part are all gathered identically.
    The random branch is the same rng call as the previous notebook -> identical indices.'''
    if n_src == n:
        return np.arange(n_src)
    if SUBSAMPLE == "fps":
        return fps_idx(xyz, n)
    return np.random.default_rng(seed).choice(n_src, n, replace=n_src < n)


_SPLIT_OFFSET = {"train": 0, "val": 500000}   # fixed, NOT hash() — python string hashing is salted


def build_split(entries, tag):
    XYZ, RGB, PART, IDS = [], [], [], []
    for i, e in enumerate(entries):
        xyz, rgb, part = load_raw(e)
        idx = subsample_idx(len(xyz), xyz, NUM_POINTS,
                            seed=SEED * 100003 + _SPLIT_OFFSET[tag] + i)
        xyz, rgb, part = xyz[idx], rgb[idx], part[idx]          # SAME indices -> correspondence kept
        XYZ.append(normalize_xyz(xyz))
        RGB.append(np.clip(rgb, 0, 1).astype(np.float32))
        PART.append(part.astype(np.int64))
        IDS.append("synthetic-%d" % e[1] if IS_SYNTHETIC else
                   os.path.splitext(os.path.basename(e))[0])
    return dict(xyz=np.stack(XYZ), rgb=np.stack(RGB), part=np.stack(PART), ids=IDS)


_perm = np.random.default_rng(SEED).permutation(len(FILES))
_tr_i = _perm[:NUM_TRAIN_SHAPES]
_va_i = _perm[NUM_TRAIN_SHAPES:NUM_TRAIN_SHAPES + NUM_VAL_SHAPES]
assert len(set(_tr_i.tolist()) & set(_va_i.tolist())) == 0, "train/val object overlap!"

t0 = time.time()
DATA = {"train": build_split([FILES[i] for i in _tr_i], "train"),
        "val":   build_split([FILES[i] for i in _va_i], "val")}
assert len(set(DATA["train"]["ids"]) & set(DATA["val"]["ids"])) == 0, "train/val id overlap!"
print(f"built in {time.time()-t0:.1f}s  |  train {DATA['train']['xyz'].shape}  "
      f"val {DATA['val']['xyz'].shape}  part {DATA['val']['part'].shape}")

for sp in ("train", "val"):
    DATA[sp]["G0"] = torch.from_numpy(DATA[sp]["xyz"]).float()
    DATA[sp]["C0"] = torch.from_numpy(DATA[sp]["rgb"]).float() * 2.0 - 1.0
    DATA[sp]["S"]  = torch.from_numpy(DATA[sp]["part"]).long()

In [ ]:
# ---------------- correspondence + normalisation diagnostics ----------------
def describe(a, name):
    a = np.asarray(a).reshape(-1, 3)
    return pd.DataFrame({"channel": list("xyz") if name.startswith(("XYZ", "G")) else list("rgb"),
                         "min": a.min(0), "max": a.max(0), "mean": a.mean(0), "std": a.std(0)}
                        ).assign(tensor=name)


_rows = []
for sp in ("train", "val"):
    _rows.append(describe(DATA[sp]["xyz"], f"XYZ unit-sphere [{sp}]"))
    _rows.append(describe(DATA[sp]["rgb"], f"RGB [0,1] [{sp}]"))
    _rows.append(describe(DATA[sp]["C0"].numpy(), f"C0 = 2*RGB-1 [{sp}]"))
NORM_STATS = pd.concat(_rows)[["tensor", "channel", "min", "max", "mean", "std"]].reset_index(drop=True)
NORM_STATS.to_csv(os.path.join(TAB_DIR, "normalization_stats.csv"), index=False)
print(NORM_STATS.to_string(index=False, float_format=lambda v: f"{v: .4f}"))

_r = np.linalg.norm(DATA["train"]["xyz"], axis=-1)
assert _r.max() <= 1.0 + 1e-5
assert DATA["train"]["C0"].abs().max() <= 1.0 + 1e-6
print(f"\nmax radius over all train points: {_r.max():.6f}  (<= 1.0, so the [-1,1] clamp is valid)")

# ---- EXPLICIT correspondence check: (G0_i, C0_i, s_i) must describe the SAME point ----
print("\ncorrespondence check — re-deriving the subsample indices and comparing:")
_ok = True
for sp, entries in (("train", [FILES[i] for i in _tr_i]), ("val", [FILES[i] for i in _va_i])):
    for i in range(min(4, len(entries))):
        rx, rc, rp = load_raw(entries[i])
        idx = subsample_idx(len(rx), rx, NUM_POINTS,
                            seed=SEED * 100003 + _SPLIT_OFFSET[sp] + i)
        same_rgb  = np.array_equal(np.clip(rc[idx], 0, 1).astype(np.float32), DATA[sp]["rgb"][i])
        same_part = np.array_equal(rp[idx].astype(np.int64), DATA[sp]["part"][i])
        same_xyz  = np.allclose(normalize_xyz(rx[idx]), DATA[sp]["xyz"][i])
        _ok &= same_rgb and same_part and same_xyz
print(f"  xyz / rgb / part all gathered with identical indices: {'PASS' if _ok else 'FAIL'}")
assert _ok, "xyz / rgb / part are not index-aligned — every part-aware result would be meaningless"

# ---- show a few aligned triples ----
_i = 0
print(f"\naligned (XYZ, RGB, part) triples for validation object [{_i}] "
      f"{DATA['val']['ids'][_i][:20]}:")
print(f"  {'idx':>5} {'x':>7} {'y':>7} {'z':>7}   {'r':>5} {'g':>5} {'b':>5}   part")
for j in np.unique(np.linspace(0, NUM_POINTS - 1, 6).astype(int)):
    x, c, p = DATA["val"]["xyz"][_i][j], DATA["val"]["rgb"][_i][j], DATA["val"]["part"][_i][j]
    nm = part_name(p)
    print(f"  {j:5d} {x[0]:7.3f} {x[1]:7.3f} {x[2]:7.3f}   {c[0]:5.2f} {c[1]:5.2f} {c[2]:5.2f}"
          f"   {p} ({nm})")

_pp = pd.DataFrame([{"object": DATA["val"]["ids"][i][:18],
                     "n_parts": len(np.unique(DATA["val"]["part"][i])),
                     **{f"part{p}": int((DATA["val"]["part"][i] == p).sum())
                        for p in range(N_PARTS_MAX)}}
                    for i in range(min(8, NUM_VAL_SHAPES))])
print("\npoints per part (first validation objects):")
print(_pp.to_string(index=False))
PART_COUNTS = np.stack([np.bincount(DATA["val"]["part"][i], minlength=N_PARTS_MAX)
                        for i in range(NUM_VAL_SHAPES)])
print(f"\nafter subsampling to {NUM_POINTS} pts: "
      f"{int((PART_COUNTS == 0).sum())} (object,part) slots are empty; smallest non-empty part has "
      f"{int(PART_COUNTS[PART_COUNTS > 0].min())} points")

## 7 · Visualisation sanity check — RGB and semantic parts side by side

Same camera, same helper as the previous notebook. Each object appears twice: once with its true
colours and once coloured by semantic part.

**Inspect:** the part map should look like a sensible decomposition (wings vs fuselage vs tail),
and — this is the point of the whole notebook — you should be able to see whether the colour map's
discontinuities line up with the part map's boundaries. §9 quantifies exactly that.

In [ ]:
def plot_cloud(ax, xyz, rgb, title="", lim=1.05, s=None, cmap_vals=None, cmap="magma",
               vmin=None, vmax=None):
    '''One 3-D panel. ShapeNet is Y-up, so we plot (x, z, y) to stand the object upright.
    Every panel in this notebook uses the same camera and the same axis limits.'''
    s = PT_SIZE if s is None else s
    if cmap_vals is None:
        sc = ax.scatter(xyz[:, 0], xyz[:, 2], xyz[:, 1], c=np.clip(rgb, 0, 1),
                        s=s, linewidths=0, depthshade=False)
    else:
        sc = ax.scatter(xyz[:, 0], xyz[:, 2], xyz[:, 1], c=cmap_vals, cmap=cmap,
                        vmin=vmin, vmax=vmax, s=s, linewidths=0, depthshade=False)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_zlim(-lim, lim)
    try:
        ax.set_box_aspect((1, 1, 1))
    except Exception:
        pass
    ax.view_init(VIEW_ELEV, VIEW_AZIM); ax.set_axis_off()
    ax.set_title(title, fontsize=8)
    return sc


def grid3d(panels, ncols=4, figsize_per=2.5, suptitle=None, path=None, **kw):
    n = len(panels); nrows = int(np.ceil(n / ncols))
    fig = plt.figure(figsize=(figsize_per * ncols, figsize_per * nrows + 0.4))
    for i, (title, xyz, rgb) in enumerate(panels):
        ax = fig.add_subplot(nrows, ncols, i + 1, projection="3d")
        plot_cloud(ax, xyz, rgb, title, **kw)
    if suptitle: fig.suptitle(suptitle, fontsize=10)
    fig.tight_layout()
    if path: fig.savefig(path, bbox_inches="tight")
    plt.show(); plt.close(fig)
    return fig


grid3d([(f"train[{i}]  {DATA['train']['ids'][i][:14]}", DATA["train"]["xyz"][i],
         DATA["train"]["rgb"][i]) for i in range(3)],
       ncols=3, suptitle=f"{CATEGORY} — normalised training clouds ({NUM_POINTS} pts)",
       path=os.path.join(FIG_DIR, "07_dataset_check.png"))

In [ ]:
_PART_PALETTE = np.array([[0.85, 0.25, 0.20], [0.20, 0.45, 0.80], [0.95, 0.75, 0.15],
                          [0.25, 0.65, 0.35], [0.60, 0.35, 0.70], [0.35, 0.75, 0.80],
                          [0.90, 0.55, 0.75], [0.45, 0.45, 0.45]])


def part_rgb(part):
    return _PART_PALETTE[np.asarray(part) % len(_PART_PALETTE)]


panels = []
for i in range(min(3, NUM_VAL_SHAPES)):
    xyz = DATA["val"]["xyz"][i]
    panels.append((f"[{i}] true RGB", xyz, DATA["val"]["rgb"][i]))
    panels.append((f"[{i}] semantic parts", xyz, part_rgb(DATA["val"]["part"][i])))
grid3d(panels, ncols=2, figsize_per=2.7,
       suptitle=f"{CATEGORY} — appearance vs semantic part structure "
                f"({' / '.join(PART_NAMES) if PART_NAMES else 'parts'})",
       path=os.path.join(FIG_DIR, "07_rgb_vs_parts.png"))
print("part colour key: " + "   ".join(
    "{}=({:.2f}, {:.2f}, {:.2f})".format(part_name(p), *_PART_PALETTE[p % len(_PART_PALETTE)])
    for p in range(N_PARTS_MAX)))

## 8 · Colour primitives (CIELAB, ΔE76, ΔE00)

Lifted unchanged from the previous notebook (`srgb_to_lab` is verbatim from
`scripts/eval_fidelity.py`). They appear *before* the models here because §9 and §10 need them to
test the hypothesis **before** anything is trained.

In [ ]:
# ---- CIELAB, verbatim from scripts/eval_fidelity.py -------------------------------------
def srgb_to_lab(rgb):
    rgb = np.clip(np.asarray(rgb, float), 0, 1)
    lin = np.where(rgb > 0.04045, ((rgb + 0.055) / 1.055) ** 2.4, rgb / 12.92)
    M = np.array([[0.4124, 0.3576, 0.1805],
                  [0.2126, 0.7152, 0.0722],
                  [0.0193, 0.1192, 0.9505]])
    xyz = (lin @ M.T) / np.array([0.95047, 1.0, 1.08883])
    d = 6 / 29
    f = np.where(xyz > d ** 3, np.cbrt(xyz), xyz / (3 * d ** 2) + 4 / 29)
    return np.stack([116 * f[:, 1] - 16, 500 * (f[:, 0] - f[:, 1]), 200 * (f[:, 1] - f[:, 2])], 1)


def delta_e76(a, b):
    return np.linalg.norm(srgb_to_lab(a) - srgb_to_lab(b), axis=1)


def delta_e00_lab(lab1, lab2):
    '''CIEDE2000 between two (N,3) Lab arrays. kL=kC=kH=1.'''
    L1, a1, b1 = lab1[:, 0], lab1[:, 1], lab1[:, 2]
    L2, a2, b2 = lab2[:, 0], lab2[:, 1], lab2[:, 2]
    C1, C2 = np.hypot(a1, b1), np.hypot(a2, b2)
    Cbar = (C1 + C2) / 2.0
    G = 0.5 * (1 - np.sqrt(Cbar ** 7 / (Cbar ** 7 + 25.0 ** 7 + 1e-30)))
    a1p, a2p = (1 + G) * a1, (1 + G) * a2
    C1p, C2p = np.hypot(a1p, b1), np.hypot(a2p, b2)
    h1p = np.degrees(np.arctan2(b1, a1p)) % 360.0
    h2p = np.degrees(np.arctan2(b2, a2p)) % 360.0
    h1p = np.where(C1p == 0, 0.0, h1p); h2p = np.where(C2p == 0, 0.0, h2p)

    dLp = L2 - L1
    dCp = C2p - C1p
    dh = h2p - h1p
    dhp = np.where(C1p * C2p == 0, 0.0,
                   np.where(np.abs(dh) <= 180, dh, np.where(dh > 180, dh - 360, dh + 360)))
    dHp = 2 * np.sqrt(C1p * C2p) * np.sin(np.radians(dhp) / 2.0)

    Lbp = (L1 + L2) / 2.0
    Cbp = (C1p + C2p) / 2.0
    hsum, hdif = h1p + h2p, np.abs(h1p - h2p)
    hbp = np.where(C1p * C2p == 0, hsum,
                   np.where(hdif <= 180, hsum / 2.0,
                            np.where(hsum < 360, (hsum + 360) / 2.0, (hsum - 360) / 2.0)))

    Tt = (1 - 0.17 * np.cos(np.radians(hbp - 30)) + 0.24 * np.cos(np.radians(2 * hbp))
          + 0.32 * np.cos(np.radians(3 * hbp + 6)) - 0.20 * np.cos(np.radians(4 * hbp - 63)))
    dtheta = 30 * np.exp(-(((hbp - 275) / 25.0) ** 2))
    Rc = 2 * np.sqrt(Cbp ** 7 / (Cbp ** 7 + 25.0 ** 7 + 1e-30))
    SL = 1 + 0.015 * (Lbp - 50) ** 2 / np.sqrt(20 + (Lbp - 50) ** 2)
    SC = 1 + 0.045 * Cbp
    SH = 1 + 0.015 * Cbp * Tt
    RT = -np.sin(np.radians(2 * dtheta)) * Rc
    return np.sqrt((dLp / SL) ** 2 + (dCp / SC) ** 2 + (dHp / SH) ** 2
                   + RT * (dCp / SC) * (dHp / SH))


def delta_e00(a_rgb, b_rgb):
    return delta_e00_lab(srgb_to_lab(a_rgb), srgb_to_lab(b_rgb))

In [ ]:
# ---------------- CIEDE2000 self-test (Sharma et al. reference pairs) ----------------
# Soft check: if it fails, deltaE00 is still reported but deltaE76 stays the primary perceptual number.
_ref = [((50.0000, 2.6772, -79.7751), (50.0000, 0.0000, -82.7485), 2.0425),
        ((50.0000, 3.1571, -77.2803), (50.0000, 0.0000, -82.7485), 2.8615),
        ((50.0000, 2.8361, -74.0200), (50.0000, 0.0000, -82.7485), 3.4412),
        ((50.0000, -1.3802, -84.2814), (50.0000, 0.0000, -82.7485), 1.0000),
        ((50.0000, 2.4900, -0.0010), (50.0000, -2.4900, 0.0009), 7.1792),
        ((60.2574, -34.0099, 36.2677), (60.4626, -34.1751, 39.4387), 1.2644),
        ((2.0776, 0.0795, -1.1350), (0.9033, -0.0636, -0.5514), 0.9082)]
_A = np.array([p[0] for p in _ref]); _B = np.array([p[1] for p in _ref])
_E = np.array([p[2] for p in _ref])
_got = delta_e00_lab(_A, _B)
DE00_OK = bool(np.abs(_got - _E).max() < 1e-2)
print("CIEDE2000 self-test:")
for g, e in zip(_got, _E):
    print(f"   computed {g:8.4f}   reference {e:8.4f}   |diff| {abs(g-e):.2e}")
print("  ->", "PASS" if DE00_OK else
      "CHECK — deltaE00 disagrees with the reference; treat deltaE76 as the primary perceptual metric")

## 9 · Hypothesis test 1 — is colour actually more coherent *within* semantic parts?

The whole architecture rests on $H(C\mid S) < H(C)$. We cannot estimate entropy directly, but we can
measure the Lab dispersion that stands in for it. For each object, with $\mu_p$ the mean Lab colour
of part $p$ and $\mu_{obj}$ the object's mean Lab colour:

$$V_p=\frac1{|S_p|}\sum_{i\in S_p}\lVert \mathrm{Lab}_i-\mu_p\rVert_2,
\qquad
V_{\text{global}}=\frac1N\sum_i\lVert \mathrm{Lab}_i-\mu_{obj}\rVert_2,$$

$$R_{\text{micro}}=\frac{\sum_p |S_p|\,V_p}{N\;V_{\text{global}}},
\qquad
R_{\text{macro}}=\frac{\frac1P\sum_p V_p}{V_{\text{global}}}.$$

$R<1$ means colour is more coherent inside parts than across the object. $R$ is reported per object,
per part and for the category as a whole, and the **full distribution is plotted — including the
objects where the assumption fails.**

A caution worth stating up front: $R$ has a floor built into it. Even for randomly assigned
"parts", splitting a cloud into $P$ groups reduces within-group dispersion somewhat, so $R<1$ alone
is weak evidence. The cell therefore also computes $R$ for **random parts of the same sizes** as a
null baseline. What matters is how far the real $R$ sits *below that null*, not below 1.

In [ ]:
def knn_np(xyz, k):
    '''XYZ-only kNN indices, self excluded. Shared by the analyses and by the metrics in §14.'''
    return cKDTree(np.asarray(xyz, float)).query(np.asarray(xyz, float), k=k + 1)[1][:, 1:]


def boundary_mask(part, nb):
    '''B = { i : some XYZ-neighbour of i carries a different semantic part label }.'''
    return (np.asarray(part)[nb] != np.asarray(part)[:, None]).any(1)


def edge_strength(rgb, knn_idx):
    '''e_i = max_{j in kNN_xyz(i)} ||c_i - c_j||_2 — local colour contrast, XYZ neighbourhoods.
    Same implementation the colour-edge metric uses in §14.'''
    return np.linalg.norm(rgb[knn_idx] - rgb[:, None, :], axis=-1).max(1)


def dispersion(lab, groups, min_pts=1):
    '''(V_p list, weights) — mean Lab distance to the group mean, per group.'''
    V, W = [], []
    for g in np.unique(groups):
        m = groups == g
        if m.sum() < min_pts:
            continue
        V.append(float(np.linalg.norm(lab[m] - lab[m].mean(0), axis=1).mean()))
        W.append(int(m.sum()))
    return np.array(V), np.array(W, float)


def coherence_row(xyz, rgb, part, rng):
    lab = srgb_to_lab(rgb)
    Vg = float(np.linalg.norm(lab - lab.mean(0), axis=1).mean())
    V, W = dispersion(lab, part, COHERENCE_MIN_PART_PTS)
    if len(V) < 2 or Vg < 1e-9:
        return None
    # null baseline: random parts with the SAME size profile
    sizes = np.bincount(part, minlength=int(part.max()) + 1)
    fake = np.repeat(np.arange(len(sizes)), sizes)[: len(part)]
    fake = rng.permutation(fake)
    Vr, Wr = dispersion(lab, fake, COHERENCE_MIN_PART_PTS)
    return dict(V_global=Vg,
                V_within_micro=float((V * W).sum() / W.sum()),
                V_within_macro=float(V.mean()),
                R_micro=float((V * W).sum() / W.sum() / Vg),
                R_macro=float(V.mean() / Vg),
                R_micro_null=float((Vr * Wr).sum() / Wr.sum() / Vg) if len(Vr) else np.nan,
                n_parts=int(len(V)))


rows, per_part = [], []
for sp in ("train", "val"):
    for i in range(len(DATA[sp]["ids"])):
        r = coherence_row(DATA[sp]["xyz"][i], DATA[sp]["rgb"][i], DATA[sp]["part"][i],
                          np.random.default_rng(7000 + i))
        if r is None:
            continue
        rows.append(dict(split=sp, shape=i, shape_id=DATA[sp]["ids"][i], **r))
        lab = srgb_to_lab(DATA[sp]["rgb"][i]); pa = DATA[sp]["part"][i]
        for p in np.unique(pa):
            m = pa == p
            if m.sum() < COHERENCE_MIN_PART_PTS:
                continue
            per_part.append(dict(split=sp, shape=i, part=int(p),
                                 name=part_name(p),
                                 n_pts=int(m.sum()),
                                 V_part=float(np.linalg.norm(lab[m] - lab[m].mean(0), axis=1).mean())))
COHERENCE = pd.DataFrame(rows)
COHERENCE_PART = pd.DataFrame(per_part)
COHERENCE.to_csv(os.path.join(TAB_DIR, "h1_coherence_per_object.csv"), index=False)
COHERENCE_PART.to_csv(os.path.join(TAB_DIR, "h1_coherence_per_part.csv"), index=False)

_r = COHERENCE.R_micro.values
_n = COHERENCE.R_micro_null.values
print(f"H1 — within-part colour coherence, {len(COHERENCE)} objects ({CATEGORY})\n")
print(f"  V_global          median {COHERENCE.V_global.median():7.3f}  (Lab units)")
print(f"  V_within (micro)  median {COHERENCE.V_within_micro.median():7.3f}")
print(f"  R_micro           median {np.median(_r):7.3f}   mean {np.nanmean(_r):.3f}   "
      f"fraction < 1: {100*np.mean(_r < 1):.0f}%")
print(f"  R_macro           median {COHERENCE.R_macro.median():7.3f}   "
      f"fraction < 1: {100*np.mean(COHERENCE.R_macro.values < 1):.0f}%")
print(f"  R_micro NULL      median {np.nanmedian(_n):7.3f}   <- random parts of the same sizes")
_gap = np.nanmedian(_n) - np.median(_r)
print(f"\n  real minus null   {_gap:+.3f}  -> semantic parts explain {_gap*100:+.1f} percentage points"
      f" of Lab dispersion\n     beyond what an arbitrary partition of the same shape would.")
print(f"\n  per-part dispersion (lower = that part is more uniformly coloured):")
print(COHERENCE_PART.groupby("name").agg(objects=("V_part", "size"), pts=("n_pts", "median"),
                                         V_part_median=("V_part", "median")).to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.0))
axes[0].hist(COHERENCE.R_micro, bins=28, color="#3b7dd8", alpha=.85, label="semantic parts")
axes[0].hist(COHERENCE.R_micro_null, bins=28, color="#e0574c", alpha=.45, label="random parts (null)")
axes[0].axvline(1.0, color="k", ls="--", lw=1)
axes[0].set_xlabel("$R_{micro}$"); axes[0].set_title("within-part / global Lab dispersion", fontsize=9)
axes[0].legend(fontsize=7)
axes[1].scatter(COHERENCE.V_global, COHERENCE.V_within_micro, s=14, alpha=.7)
_lim = [0, float(COHERENCE.V_global.max()) * 1.05]
axes[1].plot(_lim, _lim, "k--", lw=.9); axes[1].set_xlim(_lim); axes[1].set_ylim(_lim)
axes[1].set_xlabel("$V_{global}$"); axes[1].set_ylabel("$V_{within}$")
axes[1].set_title("below the diagonal = parts are more coherent", fontsize=9)
if len(COHERENCE_PART):
    order = COHERENCE_PART.groupby("name").V_part.median().sort_values().index.tolist()
    _bx = [COHERENCE_PART[COHERENCE_PART.name == n].V_part.values for n in order]
    try:                       # matplotlib >= 3.9
        axes[2].boxplot(_bx, tick_labels=order, showfliers=False)
    except TypeError:          # older matplotlib
        axes[2].boxplot(_bx, labels=order, showfliers=False)
    axes[2].set_ylabel("$V_p$ (Lab)"); axes[2].set_title("dispersion within each part", fontsize=9)
    axes[2].tick_params(axis="x", labelsize=7)
fig.suptitle(f"H1 · {CATEGORY} — is colour more coherent inside semantic parts?", fontsize=10)
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "09_h1_coherence.png")); plt.show(); plt.close(fig)

H1_SUPPORTED = bool(np.median(_r) < 1.0)
H1_BEATS_NULL = bool(np.median(_r) < np.nanmedian(_n))
print(f"H1 verdict for {CATEGORY}: R_micro median {np.median(_r):.3f} "
      f"({'<' if H1_SUPPORTED else '>='} 1) and "
      f"{'below' if H1_BEATS_NULL else 'NOT below'} the random-partition null "
      f"({np.nanmedian(_n):.3f}).")
if not H1_BEATS_NULL:
    print("  WARNING: semantic parts do no better than an arbitrary partition of the same sizes.")
    print("  The part-aware inductive bias is unmotivated for this category — expect Question B to "
          "come back null.")

## 10 · Hypothesis test 2 — are part boundaries also *colour* boundaries?

If part-aware masking is going to help, the edges it cuts should be the edges where colour actually
jumps. Using the XYZ kNN graph (`BOUNDARY_K` neighbours), split every edge $(i,j)$ into

$$\text{intra: } s_i=s_j \qquad\text{and}\qquad \text{inter: } s_i\neq s_j,$$

and compare the colour difference across them:

$$D_{\text{intra}}=\mathbb E\big[\Delta E_{00}(c_i,c_j)\mid s_i=s_j\big],\qquad
D_{\text{inter}}=\mathbb E\big[\Delta E_{00}(c_i,c_j)\mid s_i\neq s_j\big].$$

$D_{\text{inter}}/D_{\text{intra}} > 1$ means crossing a semantic boundary costs more colour error
than staying inside a part — i.e. masking those edges removes genuinely misleading context.

The cell also reports how large the boundary region is. That number matters: if a third of the
cloud sits on a boundary, hard masking is a drastic intervention, not a gentle one.

In [ ]:
rows = []
for sp in ("train", "val"):
    for i in range(len(DATA[sp]["ids"])):
        xyz, rgb, part = DATA[sp]["xyz"][i], DATA[sp]["rgb"][i], DATA[sp]["part"][i]
        if len(np.unique(part)) < 2:
            continue
        nb = knn_np(xyz, BOUNDARY_K)
        lab = srgb_to_lab(rgb)
        ci = np.repeat(np.arange(len(xyz)), BOUNDARY_K); cj = nb.ravel()
        de = delta_e00_lab(lab[ci], lab[cj])
        same = part[ci] == part[cj]
        if not same.any() or same.all():
            continue
        b = boundary_mask(part, nb)
        rows.append(dict(split=sp, shape=i, shape_id=DATA[sp]["ids"][i],
                         D_intra=float(de[same].mean()), D_inter=float(de[~same].mean()),
                         ratio=float(de[~same].mean() / max(de[same].mean(), 1e-9)),
                         boundary_frac=float(b.mean()), inter_edge_frac=float((~same).mean())))
BOUNDARY = pd.DataFrame(rows)
BOUNDARY.to_csv(os.path.join(TAB_DIR, "h2_boundary_per_object.csv"), index=False)

print(f"H2 — colour discontinuity at semantic boundaries, {len(BOUNDARY)} objects "
      f"({CATEGORY}, k={BOUNDARY_K})\n")
print(f"  D_intra  (same part)      median {BOUNDARY.D_intra.median():6.2f} ΔE00")
print(f"  D_inter  (across parts)   median {BOUNDARY.D_inter.median():6.2f} ΔE00")
print(f"  D_inter / D_intra         median {BOUNDARY.ratio.median():6.2f}   "
      f"fraction > 1: {100*np.mean(BOUNDARY.ratio.values > 1):.0f}%")
print(f"  boundary points           median {100*BOUNDARY.boundary_frac.median():5.1f}% of the cloud")
print(f"  inter-part edges          median {100*BOUNDARY.inter_edge_frac.median():5.1f}% of all edges")
H2_SUPPORTED = bool(BOUNDARY.ratio.median() > 1.0)
CAT_SUMMARY = pd.DataFrame([dict(category=CATEGORY, n_objects=len(BOUNDARY),
                                 R_micro_median=float(COHERENCE.R_micro.median()),
                                 R_micro_null_median=float(COHERENCE.R_micro_null.median()),
                                 D_intra=float(BOUNDARY.D_intra.median()),
                                 D_inter=float(BOUNDARY.D_inter.median()),
                                 ratio=float(BOUNDARY.ratio.median()),
                                 boundary_frac=float(BOUNDARY.boundary_frac.median()))])
CAT_SUMMARY.to_csv(os.path.join(TAB_DIR, "h1_h2_category_summary.csv"), index=False)
print("\ncategory-level summary:")
print(CAT_SUMMARY.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

### 10b · Do semantic boundaries coincide with *strong* colour edges?

`colour_edge_iou` in §14 measures whether a model puts colour transitions in the right places. Here
we ask the prior question: are the model's *semantic* boundaries a good proxy for where those
transitions are? Using the same GT-derived 90th-percentile colour-contrast threshold as the metric:

- **recall** = fraction of semantic-boundary points that are also strong colour-edge points;
- **precision** = fraction of strong colour-edge points that sit on a semantic boundary.

Both are compared against the chance level implied by the sizes of the two sets — a large boundary
region would score well by accident, so the raw fractions on their own mean little.

In [ ]:
rows = []
for sp in ("train", "val"):
    for i in range(len(DATA[sp]["ids"])):
        xyz, rgb, part = DATA[sp]["xyz"][i], DATA[sp]["rgb"][i], DATA[sp]["part"][i]
        if len(np.unique(part)) < 2:
            continue
        nb = knn_np(xyz, BOUNDARY_K)
        e = edge_strength(rgb, nb)
        thr = max(float(np.percentile(e, 90)), 1e-6)
        strong = e > thr
        b = boundary_mask(part, nb)
        if not strong.any() or not b.any():
            continue
        rows.append(dict(split=sp, shape=i,
                         recall=float(strong[b].mean()),
                         precision=float(b[strong].mean()),
                         chance_recall=float(strong.mean()),
                         chance_precision=float(b.mean())))
EDGE_ALIGN = pd.DataFrame(rows)
EDGE_ALIGN.to_csv(os.path.join(TAB_DIR, "h2_edge_alignment.csv"), index=False)
print(f"semantic boundary vs strong colour edge, {len(EDGE_ALIGN)} objects\n")
print(f"  recall    (boundary pts that are colour edges) : "
      f"{EDGE_ALIGN.recall.median():.3f}   chance {EDGE_ALIGN.chance_recall.median():.3f}  "
      f"-> lift x{EDGE_ALIGN.recall.median()/max(EDGE_ALIGN.chance_recall.median(),1e-9):.2f}")
print(f"  precision (colour edges on a boundary)         : "
      f"{EDGE_ALIGN.precision.median():.3f}   chance {EDGE_ALIGN.chance_precision.median():.3f}  "
      f"-> lift x{EDGE_ALIGN.precision.median()/max(EDGE_ALIGN.chance_precision.median(),1e-9):.2f}")
print("\n  A lift near 1.0 means semantic boundaries carry no more colour contrast than a random\n"
      "  set of points of the same size — parts would then be a poor proxy for appearance edges.")

fig, axes = plt.subplots(1, 3, figsize=(12, 3.0))
axes[0].hist(BOUNDARY.ratio, bins=28, color="#2f9d66", alpha=.85)
axes[0].axvline(1.0, color="k", ls="--", lw=1)
axes[0].set_xlabel(r"$D_{inter}/D_{intra}$"); axes[0].set_title("colour jump across part boundaries",
                                                                fontsize=9)
axes[1].scatter(BOUNDARY.D_intra, BOUNDARY.D_inter, s=14, alpha=.7)
_l = [0, float(max(BOUNDARY.D_inter.max(), BOUNDARY.D_intra.max())) * 1.05]
axes[1].plot(_l, _l, "k--", lw=.9); axes[1].set_xlim(_l); axes[1].set_ylim(_l)
axes[1].set_xlabel(r"$D_{intra}$ (ΔE00)"); axes[1].set_ylabel(r"$D_{inter}$ (ΔE00)")
axes[1].set_title("above the diagonal supports masking", fontsize=9)
axes[2].scatter(EDGE_ALIGN.chance_recall, EDGE_ALIGN.recall, s=14, alpha=.7, label="recall")
axes[2].scatter(EDGE_ALIGN.chance_precision, EDGE_ALIGN.precision, s=14, alpha=.7, label="precision")
_l2 = [0, 1]; axes[2].plot(_l2, _l2, "k--", lw=.9)
axes[2].set_xlabel("chance"); axes[2].set_ylabel("observed"); axes[2].legend(fontsize=7)
axes[2].set_title("boundaries vs colour edges", fontsize=9)
fig.suptitle(f"H2 · {CATEGORY} — do semantic boundaries carry colour discontinuities?", fontsize=10)
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "10_h2_boundaries.png")); plt.show(); plt.close(fig)

In [ ]:
# ---- representative visualisation: parts, boundary set, and local colour contrast ----
i = 0
xyz, rgb, part = DATA["val"]["xyz"][i], DATA["val"]["rgb"][i], DATA["val"]["part"][i]
nb = knn_np(xyz, BOUNDARY_K)
b = boundary_mask(part, nb)
e = edge_strength(rgb, nb)
fig = plt.figure(figsize=(11, 2.9))
ax = fig.add_subplot(1, 4, 1, projection="3d"); plot_cloud(ax, xyz, rgb, "true RGB")
ax = fig.add_subplot(1, 4, 2, projection="3d"); plot_cloud(ax, xyz, part_rgb(part), "semantic parts")
ax = fig.add_subplot(1, 4, 3, projection="3d")
plot_cloud(ax, xyz, np.where(b[:, None], np.array([[0.85, 0.2, 0.2]]), np.array([[0.8, 0.8, 0.85]])),
           f"semantic boundary ({100*b.mean():.0f}% of pts)")
ax = fig.add_subplot(1, 4, 4, projection="3d")
sc = plot_cloud(ax, xyz, None, "local colour contrast", cmap_vals=e, cmap="magma")
fig.colorbar(sc, ax=ax, fraction=.03)
fig.suptitle(f"object [{i}] {DATA['val']['ids'][i][:20]} — do the red points and the bright points "
             f"coincide?", fontsize=9)
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "10_boundary_example.png"),
                                bbox_inches="tight"); plt.show(); plt.close(fig)

## 8 · DDPM forward process

$$\alpha_t = 1-\beta_t,\qquad \bar\alpha_t=\prod_{s=1}^{t}\alpha_s,\qquad
x_t=\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\epsilon,\ \ \epsilon\sim\mathcal N(0,I),$$

with `T = 1000` and the cosine schedule (Nichol & Dhariwal), reusing `cosine_betas` from
`scripts/run_benchmark_kaggle.py` — the repo already uses it for low-dimensional point signals.
Training draws $t\sim\mathrm{Uniform}\{0,\dots,T-1\}$.

Experiment 1 noises **geometry only** ($C_0$ stays clean); Experiment 2 noises **colour only**
($G_0$ stays clean).

Clean-signal reconstruction:

$$\hat x_0=\frac{x_t-\sqrt{1-\bar\alpha_t}\,\hat\epsilon}{\sqrt{\bar\alpha_t}},$$

clamped to $[-1,1]$ — valid for both modalities because §6 guarantees $|G_0|\le1$ and
$C_0\in[-1,1]$. The clamp matters: at $t=900$, $\sqrt{\bar\alpha_t}$ is small enough that the
division amplifies any $\hat\epsilon$ error by a large factor, so the *unclamped* $\hat x_0$ is
meaningless. It is applied identically to both models of a pair.

**Inspect:** the four assertions below are pure numerics — they catch a broken schedule or a wrong
reconstruction formula *without training anything*. If any fails, stop and fix before §12.

In [ ]:
def make_betas(T, schedule):
    if schedule == "cosine":
        # scripts/run_benchmark_kaggle.py :: cosine_betas  (Nichol & Dhariwal, s=0.008)
        s = 0.008
        t = torch.linspace(0, T, T + 1) / T
        f = torch.cos((t + s) / (1 + s) * math.pi / 2) ** 2
        ab = f / f[0]
        return (1 - ab[1:] / ab[:-1]).clamp(1e-8, 0.999)
    if schedule == "linear":
        return torch.linspace(1e-4, 0.02, T)
    raise ValueError(schedule)


class Diffusion:
    '''Minimal DDPM forward process (no reverse sampler — this study never samples).'''

    def __init__(self, T, schedule, device):
        b = make_betas(T, schedule).to(device)
        a = 1.0 - b
        abar = torch.cumprod(a, 0)
        self.T, self.device = T, device
        self.betas, self.alphas, self.abar = b, a, abar
        self.sqrt_abar = abar.sqrt()
        self.sqrt_1mabar = (1 - abar).sqrt()

    @staticmethod
    def _ext(v, t, ndim):
        return v[t].view(-1, *([1] * (ndim - 1)))

    def q_sample(self, x0, t, noise):
        return (self._ext(self.sqrt_abar, t, x0.dim()) * x0
                + self._ext(self.sqrt_1mabar, t, x0.dim()) * noise)

    def x0_from_eps(self, x_t, t, eps, clamp=(-1.0, 1.0)):
        sa = self._ext(self.sqrt_abar, t, x_t.dim()).clamp(min=1e-5)
        sb = self._ext(self.sqrt_1mabar, t, x_t.dim())
        x0 = (x_t - sb * eps) / sa
        return x0.clamp(*clamp) if clamp is not None else x0


DIF = Diffusion(T, BETA_SCHEDULE, DEV)

# ---- assertions on the forward process (no learning involved) ----
ab = DIF.abar
assert torch.all(ab[1:] <= ab[:-1] + 1e-9), "abar must be non-increasing"
assert ab[0] > 0.99, f"abar[0] = {ab[0]:.5f} — the first step destroys too much signal"
print(f"abar[0]={ab[0]:.6f}  abar[T//2]={ab[T//2]:.6f}  abar[T-1]={ab[-1]:.3e}")

_x0 = DATA["val"]["G0"][:4].to(DEV)
_t0 = torch.zeros(4, dtype=torch.long, device=DEV)
_n = torch.randn_like(_x0)
_err0 = (DIF.q_sample(_x0, _t0, _n) - _x0).abs().max().item()
print(f"q_sample at t=0 deviates from x0 by at most {_err0:.4f}  (should be small)")
assert _err0 < 0.2

print("\nround-trip  x0 -> q_sample -> x0_from_eps(true eps)   [unclamped, float32]")
_ok = True
for tt in [0, 50, 100, 250, 500, 750, 900, 990]:
    t = torch.full((4,), tt, dtype=torch.long, device=DEV)
    xt = DIF.q_sample(_x0, t, _n)
    rec = DIF.x0_from_eps(xt, t, _n, clamp=None)
    e = (rec - _x0).abs().max().item()
    _ok &= e < 1e-2
    print(f"   t={tt:4d}  sqrt(abar)={DIF.sqrt_abar[tt]:.5f}   max|err| = {e:.2e}")
assert _ok, "clean-signal reconstruction formula is wrong"
print("\nDDPM forward process: all checks passed.")

In [ ]:
# ---- schedule diagnostic (colour only — geometry is never noised in this notebook) ----
sd_c = float(DATA["train"]["C0"].std())
snr = pd.DataFrame({"t": EVAL_TIMESTEPS})
snr["sqrt_abar"] = [float(DIF.sqrt_abar[t]) for t in EVAL_TIMESTEPS]
snr["sqrt_1-abar"] = [float(DIF.sqrt_1mabar[t]) for t in EVAL_TIMESTEPS]
snr["SNR_colour"] = snr["sqrt_abar"] * sd_c / snr["sqrt_1-abar"]
snr.to_csv(os.path.join(TAB_DIR, "snr_by_timestep.csv"), index=False)
print(f"colour signal std {sd_c:.3f}   (geometry G0 stays CLEAN throughout this notebook)\n")
print(snr.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

fig, axes = plt.subplots(1, 2, figsize=(8, 2.6))
axes[0].plot(DIF.abar.cpu()); axes[0].set_title(r"$\bar\alpha_t$"); axes[0].set_xlabel("t")
axes[1].plot(DIF.sqrt_abar.cpu(), label=r"$\sqrt{\bar\alpha_t}$")
axes[1].plot(DIF.sqrt_1mabar.cpu(), label=r"$\sqrt{1-\bar\alpha_t}$")
for t in EVAL_TIMESTEPS: axes[1].axvline(t, color="k", lw=.4, alpha=.4)
axes[1].legend(fontsize=7); axes[1].set_title("mixing coefficients (| = evaluated t)")
fig.suptitle(f"{BETA_SCHEDULE} schedule, T={T}", fontsize=10)
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "11_schedule.png")); plt.show(); plt.close(fig)

In [ ]:
# ---- what colour corruption looks like, with the part map for reference ----
_i = 0
_c0 = DATA["val"]["C0"][_i:_i + 1].to(DEV)
_gen = torch.Generator(device="cpu").manual_seed(SEED)
_nz = torch.randn(1, NUM_POINTS, 3, generator=_gen).to(DEV)
xyz0 = DATA["val"]["xyz"][_i]
pan = [("t=0 (clean)", xyz0, DATA["val"]["rgb"][_i]),
       ("semantic parts", xyz0, part_rgb(DATA["val"]["part"][_i]))]
for tt in EVAL_TIMESTEPS:
    t = torch.full((1,), tt, dtype=torch.long, device=DEV)
    ct = ((DIF.q_sample(_c0, t, _nz)[0].cpu().numpy() + 1) / 2).clip(0, 1)
    pan.append((f"$C_t$, t={tt}", xyz0, ct))
grid3d(pan, ncols=8, figsize_per=1.9,
       suptitle="colour is noised, geometry stays clean (display clipped to [0,1])",
       path=os.path.join(FIG_DIR, "11_forward_colour.png"))

## 9 · Timestep embedding

Standard sinusoidal embedding → 2-layer MLP → FiLM (per-channel scale & shift) injected into **every**
block of **all four** models. Identical mechanism, identical width, identical injection point
everywhere: the timestep pathway is never part of what differs between a baseline and its conditioned
counterpart.

Taken from `scripts/run_benchmark_kaggle.py :: timestep_embedding`.

In [ ]:
def timestep_embedding(t, dim):
    '''(B,) int timesteps -> (B, dim) sinusoidal features.'''
    half = dim // 2
    f = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / half)
    a = t.float().view(-1, 1) * f.view(1, -1)
    return torch.cat([a.sin(), a.cos()], -1)


_e = timestep_embedding(torch.arange(0, T, 10, device=DEV), WIDTH).cpu().numpy()
assert _e.shape == (T // 10, WIDTH)
assert np.abs(_e[0] - _e[-1]).max() > 0.5, "embedding does not separate t=0 from t=T"
fig, ax = plt.subplots(figsize=(6, 2.4))
im = ax.imshow(_e.T, aspect="auto", cmap="twilight", origin="lower",
               extent=[0, T, 0, WIDTH])
ax.set_xlabel("t"); ax.set_ylabel("embedding dim"); ax.set_title("sinusoidal timestep embedding")
ax.grid(False); fig.colorbar(im, ax=ax, fraction=.03)
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "09_timestep_embedding.png")); plt.show(); plt.close(fig)
print("distinct-t separation ok |", _e.shape)

## 13 · The model ladder

One class, `PointDenoiser`, instantiated five times. `C` and `C+Gpw` are structurally identical to
the previous notebook's `C` and `C+G`. The three new rungs add exactly one ingredient each.

### The part-aware aggregation

Candidates always come from XYZ:

$$\mathcal N_G(i)=\mathrm{kNN}_{XYZ}(i),\qquad
\mathcal N_P(i)=\{\,j\in\mathcal N_G(i)\;:\;s_j=s_i\,\}.$$

The EdgeConv branch computes an edge feature for **every** candidate in $\mathcal N_G(i)$ — the same
features, from the same graph, for `C+Gloc`, `C+GPid` and `C+GP`. The three differ only in what
happens next:

- `C+Gloc`, `C+GPid`: max-pool over all of $\mathcal N_G(i)$.
- `C+GP`: max-pool over $\mathcal N_P(i)$ only, via a boolean mask.

So the masking is the *single* controlled variable, exactly as required. Part labels never enter the
distance that defines $\mathcal N_G$.

**Degenerate case, handled explicitly.** If $\mathcal N_P(i)=\varnothing$ — point $i$ has no
same-part neighbour among its $k$ nearest — we do **not** silently fall back to another part's
points. Instead the block uses the point's own **self-edge** ($h_j-h_i=0$, $\mathrm{rel}=0$), which
is by definition same-part. Measured on this data that affects **~0.1 % of points**; §13c prints the
exact figure for your run. Masking uses a large finite negative value rather than $-\infty$ so no
`NaN` can enter the backward pass.

### Precomputed topology — a correctness guarantee, not just a speed-up

Geometry is never noised in this notebook, so $\mathcal N_G$, the relative-position features and the
same-part mask are **fixed for the whole run**. They are computed once per object and shared by
every model. That makes the local models nearly as fast as the pointwise ones, and it makes
"all models saw the identical graph" true by construction rather than by assumption.

In [ ]:
def _gather_nb(h, idx):                       # (B,N,C),(B,N,k) -> (B,N,k,C)
    B, N, C = h.shape
    off = (torch.arange(B, device=h.device) * N).view(B, 1, 1)
    return h.reshape(B * N, C)[(idx + off).reshape(-1)].reshape(B, N, idx.shape[-1], C)


def knn_graph(xyz, k, chunk=2048):
    '''(B,N,3) XYZ -> (B,N,k) neighbour indices, self excluded. XYZ ONLY.'''
    assert xyz.shape[-1] == 3, "neighbourhoods must be built from 3-D spatial coordinates only"
    B, N, _ = xyz.shape
    out = torch.empty(B, N, k, dtype=torch.long, device=xyz.device)
    kk = min(k + 1, N)
    for s in range(0, N, chunk):
        d = torch.cdist(xyz[:, s:s + chunk], xyz)
        idx = d.topk(kk, dim=-1, largest=False).indices[:, :, 1:]
        if idx.shape[-1] < k:
            idx = idx[..., [i % idx.shape[-1] for i in range(k)]]
        out[:, s:s + chunk] = idx
    return out


def gather_part(part, idx):                   # (B,N),(B,N,k) -> (B,N,k)
    B, N = part.shape
    return torch.gather(part, 1, idx.reshape(B, -1)).reshape(B, N, idx.shape[-1])


class Block(nn.Module):
    '''Global-context residual block, FiLM-modulated by t.
    Optional XYZ-only EdgeConv branch, optionally masked to same-part neighbours.'''

    def __init__(self, w, use_local=False, part_mask=False):
        super().__init__()
        self.use_local, self.part_mask = use_local, part_mask
        if use_local:
            self.edge = nn.Sequential(nn.Linear(2 * w + 4, w), nn.GELU(), nn.Linear(w, w))
        ctx = w * (3 if use_local else 2)
        self.fuse = nn.Sequential(nn.LayerNorm(ctx), nn.Linear(ctx, w), nn.GELU(), nn.Linear(w, w))
        self.film = nn.Linear(w, 2 * w)

    def forward(self, h, temb, ctx=None):
        g = h.max(1, keepdim=True).values.expand_as(h)
        feats = [h, g]
        if self.use_local:
            hj = _gather_nb(h, ctx["idx"]); hi = h.unsqueeze(2).expand_as(hj)
            e = self.edge(torch.cat([hi, hj - hi, ctx["rel"]], -1))          # (B,N,k,w)
            if self.part_mask:
                m = ctx["same"]                                              # (B,N,k) bool
                neg = torch.finfo(e.dtype).min / 2                           # finite -> no NaN
                agg = e.masked_fill(~m.unsqueeze(-1), neg).max(2).values
                # no same-part neighbour anywhere in kNN -> use this point's OWN self-edge,
                # never a neighbour from a different part
                z = torch.zeros_like(h)
                zr = h.new_zeros(h.shape[:-1] + (4,))
                agg = torch.where(m.any(-1, keepdim=True),
                                  agg, self.edge(torch.cat([h, z, zr], -1)))
            else:
                agg = e.max(2).values
            feats.append(agg)
        d = self.fuse(torch.cat(feats, -1))
        sc, sh = self.film(temb).unsqueeze(1).chunk(2, -1)
        return h + d * (1 + sc) + sh


class PointDenoiser(nn.Module):
    '''eps-prediction on an (N,3) per-point colour signal. One class for the whole ladder.
    part_mode: "none" (no part input) | "embed" (part embedding, unmasked kNN)
               | "mask" (part embedding AND same-part masked aggregation).'''

    def __init__(self, cond_dim=0, width=128, n_blocks=3, use_local=False, k=16,
                 part_mode="none", n_parts=8, part_emb=8):
        super().__init__()
        assert part_mode in ("none", "embed", "mask")
        self.cond_dim, self.width, self.use_local, self.k = cond_dim, width, use_local, k
        self.part_mode = part_mode
        self.uses_part = part_mode in ("embed", "mask")
        self.part_mask = (part_mode == "mask")
        self.pemb = nn.Embedding(n_parts, part_emb) if self.uses_part else None
        self.inp = nn.Linear(3 + cond_dim + (part_emb if self.uses_part else 0), width)
        self.temb = nn.Sequential(nn.Linear(width, width), nn.SiLU(), nn.Linear(width, width))
        self.blocks = nn.ModuleList([Block(width, use_local, self.part_mask)
                                     for _ in range(n_blocks)])
        self.out = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, width), nn.GELU(),
                                 nn.Linear(width, 3))

    def build_ctx(self, coords, part=None):
        '''kNN topology from XYZ ONLY; the part label only produces a MASK over those candidates.'''
        idx = knn_graph(coords, self.k)
        rel = _gather_nb(coords, idx) - coords.unsqueeze(2)
        scale = rel.norm(dim=-1).mean(dim=(1, 2), keepdim=True).clamp(min=1e-6).unsqueeze(-1)
        out = dict(idx=idx,
                   rel=torch.cat([rel / scale, rel.norm(dim=-1, keepdim=True) / scale], -1))
        out["same"] = (gather_part(part, idx) == part.unsqueeze(-1)) if part is not None else None
        return out

    def forward(self, x_t, t, cond=None, coords=None, part=None, ctx=None):
        assert (cond is None) == (self.cond_dim == 0), "cond presence must match cond_dim"
        f = [x_t] if cond is None else [x_t, cond]
        if self.uses_part:
            assert part is not None, f"{self.part_mode} model needs part labels"
            f.append(self.pemb(part))
        h = self.inp(torch.cat(f, -1))
        if t.dim() == 0:
            t = t.expand(x_t.shape[0])
        temb = self.temb(timestep_embedding(t, self.width))
        if self.use_local:
            if ctx is None:
                assert coords is not None and coords.shape[-1] == 3
                ctx = self.build_ctx(coords, part)
            assert (not self.part_mask) or ctx["same"] is not None
        for b in self.blocks:
            h = b(h, temb, ctx)
        return self.out(h)


MODEL_SPECS = {
    "C":      dict(cond=None, local=False, part_mode="none",
                   label="Model C — colour only", short="RGB"),
    "C+Gpw":  dict(cond="G0", local=False, part_mode="none",
                   label="Model C+Gpw — + clean geometry, pointwise", short="RGB+XYZ"),
    "C+Gloc": dict(cond="G0", local=True,  part_mode="none",
                   label="Model C+Gloc — + XYZ kNN aggregation", short="RGB+XYZ+kNN"),
    "C+GPid": dict(cond="G0", local=True,  part_mode="embed",
                   label="Model C+GPid — + part embedding, kNN unmasked", short="…+partID"),
    "C+GP":   dict(cond="G0", local=True,  part_mode="mask",
                   label="Model C+GP — + same-part masked aggregation", short="…+same-part kNN"),
}
ALL_COMPARISONS = [
    ("A",  "C",      "C+Gpw",  "Does clean geometry help colour denoising? (pointwise — the "
                               "previous notebook's exact comparison)"),
    ("A2", "C",      "C+Gloc", "Does clean geometry help once it is aggregated over XYZ "
                               "neighbourhoods?"),
    ("B",  "C+Gloc", "C+GP",   "Does semantic part structure add anything beyond geometry?"),
    ("B2", "C+Gloc", "C+GPid", "Does merely KNOWING the part label help?"),
    ("C",  "C+GPid", "C+GP",   "Is the gain from part identity, or from restricting information "
                               "flow to the part?"),
]
COMPARISONS = [c for c in ALL_COMPARISONS if c[1] in TRAIN_MODELS and c[2] in TRAIN_MODELS]

for _k in TRAIN_MODELS:
    assert _k in MODEL_SPECS, f"unknown model {_k}"
assert MODEL_SPECS["C"]["cond"] is None and not MODEL_SPECS["C"]["local"] \
    and MODEL_SPECS["C"]["part_mode"] == "none", "Model C must stay structurally blind"


def uses_local(key):
    return MODEL_SPECS[key]["local"]


def build_model(key):
    spec = MODEL_SPECS[key]
    set_seed(SEED)                      # identical RNG state for every model
    return PointDenoiser(cond_dim=0 if spec["cond"] is None else 3, width=WIDTH,
                         n_blocks=N_BLOCKS, use_local=spec["local"], k=PART_K,
                         part_mode=spec["part_mode"], n_parts=MAX_PARTS, part_emb=PART_EMB)


def n_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


def safe_key(k):
    return k.replace("+", "_").replace("(", "").replace(")", "")


print("model ladder:", TRAIN_MODELS)
for tag, b, c, q in COMPARISONS:
    print(f"  Question {tag:2s}: {b:7s} vs {c:7s}  — {q}")

In [ ]:
# ---------------- precompute the FIXED kNN topology (geometry is never noised here) -------------
def build_graph_cache(split, k):
    G0, S = DATA[split]["G0"], DATA[split]["S"]
    IDX, REL, SAME = [], [], []
    probe = PointDenoiser(cond_dim=3, width=8, n_blocks=1, use_local=True, k=k)
    for i in range(len(G0)):
        ctx = probe.build_ctx(G0[i:i + 1], S[i:i + 1])
        IDX.append(ctx["idx"][0]); REL.append(ctx["rel"][0]); SAME.append(ctx["same"][0])
    return dict(idx=torch.stack(IDX), rel=torch.stack(REL), same=torch.stack(SAME))


t0 = time.time()
GRAPH = {sp: build_graph_cache(sp, PART_K) for sp in ("train", "val")}
print(f"precomputed kNN topology for {len(GRAPH['train']['idx'])} train + "
      f"{len(GRAPH['val']['idx'])} val objects in {time.time()-t0:.1f}s  (k={PART_K})")


def ctx_for(split, idx_batch):
    '''Slice the cached graph for a batch and move it to the device. Shared by ALL models.'''
    g = GRAPH[split]
    return dict(idx=g["idx"][idx_batch].to(DEV), rel=g["rel"][idx_batch].to(DEV),
                same=g["same"][idx_batch].to(DEV))


# ---- how often does same-part masking hit the degenerate case? ----
for sp in ("train", "val"):
    sm = GRAPH[sp]["same"]
    n_same = sm.sum(-1).float()
    print(f"  [{sp}] same-part neighbours per point: mean {n_same.mean():.2f}/{PART_K}   "
          f"points with ZERO: {100*(n_same == 0).float().mean():.3f}%   "
          f"with < k/2: {100*(n_same < PART_K/2).float().mean():.2f}%")
FALLBACK_FRAC = float((GRAPH["val"]["same"].sum(-1) == 0).float().mean())
print(f"  -> the self-edge fallback fires for {FALLBACK_FRAC*100:.3f}% of validation points")

In [ ]:
# ---------------- parameter counts + shape smoke test ----------------
_probe = {k: build_model(k).to(DEV) for k in TRAIN_MODELS}
_B = 2
_g0 = DATA["val"]["G0"][:_B].to(DEV)
_c0 = DATA["val"]["C0"][:_B].to(DEV)
_s0 = DATA["val"]["S"][:_B].to(DEV)
_ctx = ctx_for("val", np.arange(_B))
_t = torch.full((_B,), 300, dtype=torch.long, device=DEV)


def _fwd(model, key, x_t, t, G0, C0, S, ctx):
    '''Assemble the call for one model. cond / part / ctx are supplied only where the spec allows.'''
    spec = MODEL_SPECS[key]
    cond = None if spec["cond"] is None else (G0 if spec["cond"] == "G0" else C0)
    part = S if spec["part_mode"] != "none" else None
    return model(x_t, t, cond, part=part, ctx=(ctx if spec["local"] else None))


rows = []
for k, m in _probe.items():
    with torch.no_grad():
        o = _fwd(m, k, _c0, _t, _g0, _c0, _s0, _ctx)
    assert o.shape == (_B, NUM_POINTS, 3), f"{k}: expected (B,N,3), got {tuple(o.shape)}"
    sp = MODEL_SPECS[k]
    rows.append(dict(model=k, description=sp["label"], input=sp["short"],
                     geometry=sp["cond"] is not None, knn=sp["local"],
                     part_mode=sp["part_mode"], params=n_params(m), out=str(tuple(o.shape))))
PARAM_TABLE = pd.DataFrame(rows)
PARAM_TABLE.to_csv(os.path.join(TAB_DIR, "model_parameters.csv"), index=False)
print(PARAM_TABLE.to_string(index=False))
print()
for tag, b, c, _q in COMPARISONS:
    pb = int(PARAM_TABLE.set_index("model").loc[b, "params"])
    pc = int(PARAM_TABLE.set_index("model").loc[c, "params"])
    print(f"  Q{tag:2s}  {b:7s} {pb:,} vs {c:7s} {pc:,}  ->  {100*(pc-pb)/pb:+.2f}% parameters")
print("\nC+GPid and C+GP are parameter-IDENTICAL — they differ only in whether the aggregation is\n"
      "masked, which is exactly what Question C is meant to isolate.")

### 13c · Structural tests — extended for part labels

The previous notebook's leakage test is reused and widened. With $x_t$ and the kNN topology held
fixed, each auxiliary input is perturbed in turn and we check the output moves **only** where the
model definition allows:

| model | geometry perturbed | part labels permuted |
|---|---|---|
| `C` | must not move | must not move |
| `C+Gpw`, `C+Gloc` | must move | **must not move** |
| `C+GPid`, `C+GP` | must move | must move |

Two further checks that matter specifically for the part-aware claim:

1. **Topology invariance** — permuting part labels must leave the kNN *indices* bit-identical.
   If it didn't, part labels would be redefining geometry, which rule 2 forbids.
2. **Mask purity** — every neighbour actually used by the masked aggregation must carry the same
   part label as its centre point. This is asserted directly on the mask tensor.

In [ ]:
def leakage_table(models, n=4, t_val=300):
    g0 = DATA["val"]["G0"][:n].to(DEV); c0 = DATA["val"]["C0"][:n].to(DEV)
    s0 = DATA["val"]["S"][:n].to(DEV)
    ctx = ctx_for("val", np.arange(n))
    tv = torch.full((n,), t_val, dtype=torch.long, device=DEV)
    gen = torch.Generator().manual_seed(11)
    nz = torch.randn(n, NUM_POINTS, 3, generator=gen).to(DEV)
    x_t = DIF.q_sample(c0, tv, nz)

    Rm = torch.linalg.qr(torch.randn(3, 3, generator=torch.Generator().manual_seed(7)))[0].to(DEV)
    g0_alt = g0 @ Rm                                    # rotated, equally valid geometry
    # a PERMUTATION of the label ids: same partition of the cloud, different names for the parts
    lut = torch.tensor([1, 0, 3, 2, 4, 5, 6, 7][:MAX_PARTS], device=DEV)
    s0_alt = lut[s0]
    ctx_alt = dict(idx=ctx["idx"], rel=ctx["rel"],
                   same=(gather_part(s0_alt, ctx["idx"]) == s0_alt.unsqueeze(-1)))
    assert torch.equal(ctx_alt["same"], ctx["same"]), \
        "relabelling parts changed the mask — the lut must be a bijection"

    rows = []
    for k, m in models.items():
        spec = MODEL_SPECS[k]
        with torch.no_grad():
            base = _fwd(m, k, x_t, tv, g0, c0, s0, ctx)
            d_g = (_fwd(m, k, x_t, tv, g0_alt, c0, s0, ctx) - base).abs().max().item()
            d_s = (_fwd(m, k, x_t, tv, g0, c0, s0_alt, ctx_alt) - base).abs().max().item()
        exp_g = "uses" if spec["cond"] is not None else "blind"
        exp_s = "uses" if spec["part_mode"] != "none" else "blind"
        ok = ((d_g > 0) if exp_g == "uses" else (d_g == 0.0)) and \
             ((d_s > 0) if exp_s == "uses" else (d_s == 0.0))
        rows.append(dict(model=k, expect_geometry=exp_g, expect_part=exp_s,
                         d_out_perturb_G0=d_g, d_out_perturb_S=d_s,
                         verdict="PASS" if ok else "FAIL"))
    return pd.DataFrame(rows)


print("leakage test — x_t and kNN topology held fixed, each auxiliary input perturbed in turn:\n")
LEAK_TABLE = leakage_table(_probe)
LEAK_TABLE.to_csv(os.path.join(TAB_DIR, "leakage_test.csv"), index=False)
print(LEAK_TABLE.to_string(index=False))
assert (LEAK_TABLE.verdict == "PASS").all(), "modality leakage detected — do not trust any result"

# ---- 1. topology invariance: part labels must NOT influence the kNN indices ----
_pm = PointDenoiser(cond_dim=3, width=16, n_blocks=1, use_local=True, k=PART_K,
                    part_mode="mask", n_parts=MAX_PARTS, part_emb=PART_EMB)
_ctx_a = _pm.build_ctx(_g0, _s0)
_ctx_b = _pm.build_ctx(_g0, (_s0 + 1) % MAX_PARTS)
assert torch.equal(_ctx_a["idx"], _ctx_b["idx"]), "part labels changed the kNN graph!"
assert torch.equal(_ctx_a["rel"], _ctx_b["rel"]), "part labels changed the relative features!"
print("\ntopology invariance: changing part labels leaves kNN indices and rel features "
      "bit-identical  PASS")

# ---- 2. mask purity: no neighbour used by the masked aggregation crosses a part boundary ----
_bad = 0
for sp in ("train", "val"):
    idx, same = GRAPH[sp]["idx"], GRAPH[sp]["same"]
    S = DATA[sp]["S"]
    nbp = torch.gather(S, 1, idx.reshape(len(S), -1)).reshape(same.shape)
    _bad += int((same & (nbp != S.unsqueeze(-1))).sum())
assert _bad == 0, f"{_bad} masked-in neighbours carry a different part label"
print(f"mask purity: 0 of {int(GRAPH['train']['same'].numel() + GRAPH['val']['same'].numel()):,} "
      f"candidate edges admitted by the same-part mask crosses a part boundary  PASS")
_kept = float(GRAPH["val"]["same"].float().mean())
print(f"the mask keeps {_kept*100:.1f}% of candidate edges and discards {100-_kept*100:.1f}% "
      f"as cross-part.")
del _probe, _pm

## 14 · Metrics — the full previous suite, plus part-aware and boundary metrics

**Everything from the previous notebook is kept unchanged** (`rgb_mae`, `rgb_mse`, `psnr`,
`deltaE76`, `deltaE00`, `deltaE76_p95`, `local_consistency_absdiff`/`_ratio`, `hist_chi2`,
`swd_lab`, `region_dE`, `region_dE_norm`, `colour_edge_iou`) so results stay directly comparable.
Four families are added:

**A · Part-macro ΔE00** — $E_p=\frac1{|S_p|}\sum_{i\in S_p}\Delta E_{00}(\hat c_i,c_i)$, then
$E_{macro}=\frac1P\sum_p E_p$. A fuselage covering half the points cannot drown out a badly
reconstructed engine. The ordinary point-weighted (micro) value is `deltaE00`, reported alongside.

**B · Per-part Lab SWD** — the notebook's existing sliced-Wasserstein, applied *within* each part
and macro-averaged. Asks whether each part's colour *distribution* is right, not just its mean.

**C · Per-part local consistency** — the existing local-roughness measure restricted to same-part
XYZ neighbours. Separates over-smoothing from speckle *inside* a part, where masking should act.

**D · Boundary vs interior ΔE00** — with $B=\{i:\exists j\in\mathcal N_G(i),\,s_j\neq s_i\}$ and
$I$ its complement, report $\Delta E_{00}^{B}$, $\Delta E_{00}^{I}$ and their ratio **separately**.
This is the metric that would expose the obvious failure mode of hard masking: cleaner part
interiors bought at the cost of wrecked transitions. If `C+GP` improves the interior while degrading
the boundary, that shows up here and nowhere else.

In [ ]:
# ---------------- helpers (unchanged from the previous notebook) ----------------
def local_roughness(rgb, knn_idx):
    '''s(C) = mean_i mean_{j in kNN_xyz(i)} ||c_i - c_j||_2 ; neighbourhoods from XYZ only.'''
    return float(np.linalg.norm(rgb[knn_idx] - rgb[:, None, :], axis=-1).mean())


def hist_chi2(a, b, bins=8):
    ha = np.histogramdd(np.clip(a, 0, 1), bins=bins, range=[(0, 1)] * 3)[0].ravel()
    hb = np.histogramdd(np.clip(b, 0, 1), bins=bins, range=[(0, 1)] * 3)[0].ravel()
    ha = ha / max(ha.sum(), 1); hb = hb / max(hb.sum(), 1)
    return 0.5 * float(np.sum((ha - hb) ** 2 / (ha + hb + 1e-12)))


_SWD_DIRS = None
def swd(A, B, n_proj=64):
    '''Sliced Wasserstein-1 between two equal-size point sets, FIXED projections notebook-wide.'''
    global _SWD_DIRS
    if _SWD_DIRS is None:
        P = np.random.default_rng(12345).normal(size=(3, n_proj))
        _SWD_DIRS = P / np.linalg.norm(P, axis=0, keepdims=True)
    a = np.sort(A @ _SWD_DIRS, axis=0); b = np.sort(B @ _SWD_DIRS, axis=0)
    return float(np.abs(a - b).mean())


class ShapeCtx:
    '''Everything a metric needs about one validation object. Depends ONLY on ground truth, so it is
    identical for every model — that is what makes the comparison paired. Extended with the
    semantic part label, the boundary/interior split and the same-part neighbour mask.'''

    def __init__(self, xyz, rgb, part, k=8, n_regions=32, seed=0, bnd_k=8):
        self.xyz, self.rgb = xyz.astype(np.float64), rgb.astype(np.float64)
        self.part = np.asarray(part).astype(np.int64)
        self.lab = srgb_to_lab(self.rgb)
        self.knn = knn_np(self.xyz, k)                                    # XYZ only
        self.rough_gt = local_roughness(self.rgb, self.knn)
        seeds = fps_idx(self.xyz.astype(np.float32), min(n_regions, len(xyz)))
        self.cell = cdist(self.xyz, self.xyz[seeds]).argmin(1)
        self.cell_w = np.bincount(self.cell, minlength=len(seeds)).astype(float)
        self.n_cells = len(seeds)
        self.lab_cell_gt = srgb_to_lab(self.cell_means(self.rgb))
        self.edge_gt = edge_strength(self.rgb, self.knn)
        self.edge_thr = max(float(np.percentile(self.edge_gt, 90)), 1e-6)
        self.edge_mask_gt = self.edge_gt > self.edge_thr
        perm = np.random.default_rng(seed).permutation(len(xyz))
        self.region_chance = self.region_dE(self.rgb[perm])
        # ---- part-aware additions ----
        self.nb_bnd = knn_np(self.xyz, bnd_k)                             # XYZ only
        self.boundary = boundary_mask(self.part, self.nb_bnd)
        self.interior = ~self.boundary
        self.same_knn = self.part[self.knn] == self.part[:, None]         # (N,k)
        self.part_ids = [int(p) for p in np.unique(self.part)]
        self.part_n = {p: int((self.part == p).sum()) for p in self.part_ids}
        self.rough_part_gt = self.part_roughness(self.rgb)
        V, W = dispersion(self.lab, self.part, COHERENCE_MIN_PART_PTS)
        Vg = float(np.linalg.norm(self.lab - self.lab.mean(0), axis=1).mean())
        self.coherence_R = float((V * W).sum() / W.sum() / Vg) if (len(V) >= 2 and Vg > 1e-9) \
            else float("nan")

    # ---- unchanged ----
    def cell_means(self, rgb):
        s = np.zeros((self.n_cells, 3)); np.add.at(s, self.cell, rgb)
        return s / np.maximum(self.cell_w, 1)[:, None]

    def region_dE(self, rgb):
        d = np.linalg.norm(srgb_to_lab(self.cell_means(rgb)) - self.lab_cell_gt, axis=1)
        return float(np.average(d, weights=self.cell_w))

    def edge_iou(self, rgb):
        mp = edge_strength(rgb, self.knn) > self.edge_thr
        union = int((mp | self.edge_mask_gt).sum())
        if not self.edge_mask_gt.any() or union == 0:
            return float("nan")
        return float((mp & self.edge_mask_gt).sum() / union)

    # ---- part-aware ----
    def part_roughness(self, rgb):
        '''Local colour roughness restricted to SAME-PART XYZ neighbours.'''
        d = np.linalg.norm(rgb[self.knn] - rgb[:, None, :], axis=-1)
        cnt = self.same_knn.sum(1)
        ok = cnt > 0
        if not ok.any():
            return float("nan")
        return float((np.where(self.same_knn, d, 0.0).sum(1)[ok] / cnt[ok]).mean())

    def per_part_dE00(self, lab_pred):
        return {p: float(delta_e00_lab(self.lab[self.part == p],
                                       lab_pred[self.part == p]).mean()) for p in self.part_ids}

    def part_macro_dE00(self, lab_pred):
        v = list(self.per_part_dE00(lab_pred).values())
        return float(np.mean(v)) if v else float("nan")

    def part_swd_macro(self, lab_pred, min_pts=4):
        v = [swd(lab_pred[self.part == p], self.lab[self.part == p])
             for p in self.part_ids if self.part_n[p] >= min_pts]
        return float(np.mean(v)) if v else float("nan")


def colour_metrics(ctx, rgb_pred, per_point=False):
    rgb_pred = np.clip(np.asarray(rgb_pred, float), 0, 1)
    lab_p = srgb_to_lab(rgb_pred)
    d76 = np.linalg.norm(lab_p - ctx.lab, axis=1)
    d00 = delta_e00_lab(ctx.lab, lab_p)
    err = rgb_pred - ctx.rgb
    mse = float((err ** 2).mean())
    rough_p = local_roughness(rgb_pred, ctx.knn)
    reg = ctx.region_dE(rgb_pred)
    reg_norm = reg / ctx.region_chance if ctx.region_chance > 1.0 else float("nan")
    rp = ctx.part_roughness(rgb_pred)
    dB = float(d00[ctx.boundary].mean()) if ctx.boundary.any() else float("nan")
    dI = float(d00[ctx.interior].mean()) if ctx.interior.any() else float("nan")
    out = dict(
        # ---- previous notebook's suite, unchanged ----
        rgb_mae=float(np.abs(err).mean()), rgb_mse=mse,
        psnr=float(10 * np.log10(1.0 / max(mse, 1e-12))),
        deltaE76=float(d76.mean()), deltaE00=float(d00.mean()),
        deltaE76_p95=float(np.percentile(d76, 95)),
        local_consistency_absdiff=abs(rough_p - ctx.rough_gt),
        local_consistency_ratio=(rough_p / ctx.rough_gt if ctx.rough_gt > 1e-6 else float("nan")),
        hist_chi2=hist_chi2(rgb_pred, ctx.rgb), swd_lab=swd(lab_p, ctx.lab),
        region_dE=reg, region_dE_norm=reg_norm, region_dE_chance=ctx.region_chance,
        colour_edge_iou=ctx.edge_iou(rgb_pred),
        # ---- part-aware additions ----
        part_macro_dE00=ctx.part_macro_dE00(lab_p),
        part_swd_macro=ctx.part_swd_macro(lab_p),
        part_consistency_absdiff=(abs(rp - ctx.rough_part_gt) if rp == rp else float("nan")),
        part_consistency_ratio=(rp / ctx.rough_part_gt if ctx.rough_part_gt > 1e-6 else float("nan")),
        boundary_dE00=dB, interior_dE00=dI,
        boundary_interior_ratio=(dB / dI if (dI == dI and dI > 1e-9) else float("nan")))
    if per_point:
        out["_dE76_pp"], out["_dE00_pp"] = d76, d00
    return out


HIGHER_IS_BETTER = {"psnr", "colour_edge_iou"}
TARGET_ONE = {"local_consistency_ratio", "part_consistency_ratio"}
COL_METRICS = ["eps_mse", "rgb_mae", "rgb_mse", "psnr", "deltaE76", "deltaE00",
               "local_consistency_absdiff", "hist_chi2", "swd_lab",
               "region_dE", "region_dE_norm", "colour_edge_iou"]
PART_METRICS = ["part_macro_dE00", "part_swd_macro", "part_consistency_absdiff",
                "boundary_dE00", "interior_dE00", "boundary_interior_ratio"]
ALL_METRICS = COL_METRICS + PART_METRICS
HEADLINE_METRICS = ["eps_mse", "deltaE00", "part_macro_dE00", "part_swd_macro", "region_dE",
                    "boundary_dE00", "interior_dE00", "colour_edge_iou"]


def direction(m):
    return "higher" if m in HIGHER_IS_BETTER else ("target1" if m in TARGET_ONE else "lower")


def arrow(m):
    return {"higher": "↑", "lower": "↓", "target1": "→1"}[direction(m)]


t0 = time.time()
VAL_CTX = [ShapeCtx(DATA["val"]["xyz"][i], DATA["val"]["rgb"][i], DATA["val"]["part"][i],
                    k=KNN_K, n_regions=N_REGIONS, seed=1000 + i, bnd_k=BOUNDARY_K)
           for i in range(NUM_VAL_SHAPES)]
print(f"built {len(VAL_CTX)} validation contexts in {time.time()-t0:.1f}s")
_bf = np.array([c.boundary.mean() for c in VAL_CTX])
print(f"boundary points: median {100*np.median(_bf):.1f}% of the cloud "
      f"(min {100*_bf.min():.1f}%, max {100*_bf.max():.1f}%)")
print(f"parts per validation object: {np.bincount([len(c.part_ids) for c in VAL_CTX])[1:].tolist()} "
      f"(index = 1,2,3,... parts)")

# ---- self-consistency: the ground truth must score perfectly on EVERY metric ----
def _isnan(v):
    return v != v


for _c in VAL_CTX:
    _m = colour_metrics(_c, _c.rgb)
    assert _m["rgb_mae"] == 0 and _m["deltaE76"] == 0 and _m["deltaE00"] == 0, _m
    assert _m["region_dE"] < 1e-9 and _m["hist_chi2"] < 1e-12 and _m["swd_lab"] < 1e-9, _m
    assert _m["local_consistency_absdiff"] == 0.0, _m
    assert _m["part_macro_dE00"] < 1e-9 and _m["part_swd_macro"] < 1e-9, _m
    assert _isnan(_m["part_consistency_absdiff"]) or _m["part_consistency_absdiff"] < 1e-12, _m
    assert _isnan(_m["boundary_dE00"]) or _m["boundary_dE00"] < 1e-9, _m
    assert _isnan(_m["interior_dE00"]) or _m["interior_dE00"] < 1e-9, _m
    assert _m["colour_edge_iou"] == 1.0 or (_isnan(_m["colour_edge_iou"])
                                            and not _c.edge_mask_gt.any()), _m
print("metric self-consistency on an exact prediction: PASS for all "
      f"{len(VAL_CTX)} objects, on the part and boundary metrics too")

## 15 · Tiny-set overfit sanity check  ← **read the verdict before going further**

Identical protocol and identical pass criteria to the previous notebook, now run for all five
models. Reminder of why $\epsilon$-MSE alone is not the criterion: the task gets *easier* as $t$
rises (at high $t$, $\epsilon\approx x_t$, so a net can echo its input), so the uniform-$t$ average
is dominated by the trivial regime. The decisive criterion is the **$x_0$ recovery gain** at
$t=250$ — the fraction of the naive "assume it is not noisy" error the model actually removes.

**Pass criteria** (all four, for every model): loss below 70 % of its opening level; final loss
below 0.85 (clear of the $\hat\epsilon\equiv0$ predictor, which scores exactly 1.0); the
$t{=}750 < t{=}250 < t{=}50$ difficulty profile; and $x_0$ gain at $t{=}250$ above 0.15.

In [ ]:
SANITY_LR = 1e-3
PROBE_TS  = (50, 250, 750)


def tiny_overfit(key, n_clouds=None, steps=None, lr=SANITY_LR, log_every=25):
    n_clouds = SANITY_CLOUDS if n_clouds is None else n_clouds
    steps = SANITY_STEPS if steps is None else steps
    set_seed(SEED)
    model = build_model(key).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)
    sel = np.arange(n_clouds)
    G0 = DATA["train"]["G0"][sel].to(DEV)
    C0 = DATA["train"]["C0"][sel].to(DEV)
    S  = DATA["train"]["S"][sel].to(DEV)
    ctx = ctx_for("train", sel)
    x0 = C0

    pg = torch.Generator().manual_seed(4242)
    probe_noise = torch.randn(n_clouds, NUM_POINTS, 3, generator=pg).to(DEV)
    losses, probe = [], {t: [] for t in PROBE_TS}
    for s in range(steps):
        g = torch.Generator().manual_seed(SEED * 7919 + s)
        t = torch.randint(0, T, (n_clouds,), generator=g).to(DEV)
        noise = torch.randn(n_clouds, NUM_POINTS, 3, generator=g).to(DEV)
        model.train()
        loss = F.mse_loss(_fwd(model, key, DIF.q_sample(x0, t, noise), t, G0, C0, S, ctx), noise)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
        if s % log_every == 0 or s == steps - 1:
            model.eval()
            with torch.no_grad():
                for tt in PROBE_TS:
                    tv = torch.full((n_clouds,), tt, dtype=torch.long, device=DEV)
                    x_t = DIF.q_sample(x0, tv, probe_noise)
                    eps = _fwd(model, key, x_t, tv, G0, C0, S, ctx)
                    probe[tt].append((s, F.mse_loss(eps, probe_noise).item(),
                                      F.mse_loss(DIF.x0_from_eps(x_t, tv, eps), x0).item(),
                                      F.mse_loss(DIF.x0_from_eps(x_t, tv, torch.zeros_like(eps)),
                                                 x0).item()))
    return dict(losses=np.array(losses), probe={k: np.array(v) for k, v in probe.items()})


def probe_last(h, tt, col):
    return float(h["probe"][tt][-1, col])


def x0_gain(h, tt):
    triv = probe_last(h, tt, 3)
    return float("nan") if triv <= 0 else 1.0 - probe_last(h, tt, 2) / triv


_OPEN = min(5, SANITY_STEPS)
_EDGE = max(10, SANITY_STEPS // 20)
if RUN_SANITY_CHECK:
    SANITY = {}
    SANITY_SECONDS = {}
    for k in TRAIN_MODELS:
        t0 = time.time()
        SANITY[k] = tiny_overfit(k)
        SANITY_SECONDS[k] = time.time() - t0
        L = SANITY[k]["losses"]
        print(f"  {k:7s} {SANITY_SECONDS[k]:5.1f}s   loss {L[:_OPEN].mean():.4f} -> "
              f"{L[-_EDGE:].mean():.4f}", flush=True)
else:
    SANITY = SANITY_SECONDS = None
    print("RUN_SANITY_CHECK is False — skipped.")

In [ ]:
if SANITY is not None:
    rows = []
    for k, h in SANITY.items():
        init = float(h["losses"][:_OPEN].mean()); fin = float(h["losses"][-_EDGE:].mean())
        p50, p250, p750 = (probe_last(h, t, 1) for t in PROBE_TS)
        gain = x0_gain(h, 250)
        c1, c2, c3, c4 = fin < 0.70 * init, fin < 0.85, p750 < p250 < p50, gain > 0.15
        rows.append(dict(model=k, loss_start=init, loss_end=fin, ratio=fin / init,
                         eps_t50=p50, eps_t250=p250, eps_t750=p750,
                         x0_gain_t50=x0_gain(h, 50), x0_gain_t250=gain,
                         drop_ok=c1, beats_trivial_ok=c2, t_profile_ok=c3, x0_gain_ok=c4,
                         passed=bool(c1 and c2 and c3 and c4)))
    SANITY_TABLE = pd.DataFrame(rows)
    SANITY_TABLE.to_csv(os.path.join(TAB_DIR, "sanity_check.csv"), index=False)
    print(SANITY_TABLE.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    print("\neps-MSE reference: predicting eps = 0 scores exactly 1.0 at EVERY t, and eps-MSE is\n"
          "expected to FALL as t rises — which is why the x0 gain is the criterion that matters.")

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.1))
    for k, h in SANITY.items():
        w = max(1, len(h["losses"]) // 60)
        axes[0].plot(np.convolve(h["losses"], np.ones(w) / w, mode="valid"), lw=1.2, label=k)
        axes[1].plot(h["probe"][50][:, 0], h["probe"][50][:, 1], lw=1.2, label=k)
        axes[2].plot(h["probe"][250][:, 0],
                     1 - h["probe"][250][:, 2] / h["probe"][250][:, 3], lw=1.2, label=k)
    axes[0].axhline(1.0, color="k", ls="--", lw=.8); axes[0].axhline(0.85, color="g", ls=":", lw=.8)
    axes[0].set_title("tiny-set loss (uniform t)"); axes[0].set_xlabel("step")
    axes[0].set_ylabel(r"$\epsilon$-MSE"); axes[0].legend(fontsize=7)
    axes[1].axhline(1.0, color="k", ls="--", lw=.8)
    axes[1].set_title(r"$\epsilon$-MSE at t=50 (the hard end)"); axes[1].legend(fontsize=7)
    axes[2].axhline(0.15, color="g", ls=":", lw=.8); axes[2].axhline(0.0, color="k", lw=.8)
    axes[2].set_title(r"$x_0$ recovery gain at t=250 (pass > 0.15)"); axes[2].legend(fontsize=7)
    fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "15_sanity.png")); plt.show(); plt.close(fig)

    SANITY_PASSED = bool(SANITY_TABLE.passed.all())
    print("\n" + "=" * 74)
    if SANITY_PASSED:
        print("SANITY CHECK PASSED")
        print(f"  All {len(TRAIN_MODELS)} models drive the denoising loss down on a tiny "
              f"memorised set.")
        print("  -> set RUN_FULL_EXPERIMENT = True in §1, re-run §1, then run §16 onwards.")
    else:
        print("SANITY CHECK FAILED — inspect loss curve before proceeding")
        print(f"  failing models: {SANITY_TABLE.loc[~SANITY_TABLE.passed, 'model'].tolist()}")
        print("  Do NOT enable RUN_FULL_EXPERIMENT. Debug order: §11 assertions -> §6 "
              "normalisation -> raise SANITY_STEPS or WIDTH.")
    print("=" * 74)

## 16 · Training

The previous notebook's protocol, unchanged, extended to the five-model ladder:

* **identical batches** — the epoch permutation comes from `default_rng([SEED, epoch])`;
* **identical timesteps and noise** — step *(e,s)* seeds a fresh generator from `(SEED, e, s)`, so
  every model trains on byte-identical $(C_t, t, \epsilon)$ triples;
* **identical topology** — all local models read the same precomputed kNN cache;
* **identical initial RNG state** — `build_model` re-seeds before construction;
* **one frozen validation bank** shared by every model, sampled every `VAL_EVERY` epochs (and
  always on the final epoch, so the reported end-of-training number is exact).

Per-epoch checkpoints are resumable and are rejected if the stored config no longer matches §1.
The cell below prints a **runtime estimate extrapolated from this session's §15 timings** before
anything is trained — read it before committing to the run.

In [ ]:
def epoch_batches(n, bs, epoch):
    perm = np.random.default_rng([SEED, epoch]).permutation(n)
    return [perm[i:i + bs] for i in range(0, n, bs)]


def draw_t_noise(bs, epoch, step):
    g = torch.Generator().manual_seed((SEED * 1000003 + epoch * 10007 + step) % (2 ** 31 - 1))
    t = torch.randint(0, T, (bs,), generator=g)
    noise = torch.randn(bs, NUM_POINTS, 3, generator=g)
    return t.to(DEV), noise.to(DEV)


def build_val_bank(repeats, seed):
    n, M = NUM_VAL_SHAPES, NUM_VAL_SHAPES * repeats
    g = torch.Generator().manual_seed(seed)
    u = (torch.arange(M).float() + torch.rand(M, generator=g)) / M
    t = (u * T).long().clamp(0, T - 1)
    p = torch.randperm(M, generator=g)
    return dict(t=t[p].to(DEV), noise=torch.randn(M, NUM_POINTS, 3, generator=g).to(DEV),
                shape=torch.arange(n).repeat(repeats))


VAL_BANK = build_val_bank(VAL_BANK_REPEATS, seed=SEED + 555)
print(f"frozen val bank: {len(VAL_BANK['t'])} (shape, t, noise) triples, "
      f"t spans {int(VAL_BANK['t'].min())}..{int(VAL_BANK['t'].max())}")


@torch.no_grad()
def val_loss(model, key, bank=None, chunk=16):
    bank = VAL_BANK if bank is None else bank
    model.eval(); tot, cnt = 0.0, 0
    for s in range(0, len(bank["t"]), chunk):
        sl = slice(s, s + chunk)
        idx = bank["shape"][sl].numpy()
        G0 = DATA["val"]["G0"][idx].to(DEV); C0 = DATA["val"]["C0"][idx].to(DEV)
        S = DATA["val"]["S"][idx].to(DEV); ctx = ctx_for("val", idx)
        t, noise = bank["t"][sl], bank["noise"][sl]
        eps = _fwd(model, key, DIF.q_sample(C0, t, noise), t, G0, C0, S, ctx)
        tot += F.mse_loss(eps, noise, reduction="sum").item(); cnt += noise.numel()
    return tot / cnt


def _train_meta():
    return dict(seed=SEED, category=CATEGORY, n_points=NUM_POINTS, n_train=NUM_TRAIN_SHAPES,
                n_val=NUM_VAL_SHAPES, bs=BATCH_SIZE, lr=LEARNING_RATE, wd=WEIGHT_DECAY,
                epochs=NUM_EPOCHS, T=T, schedule=BETA_SCHEDULE, width=WIDTH, blocks=N_BLOCKS,
                lr_sched=LR_SCHEDULE, source=DATA_ORIGIN, subsample=SUBSAMPLE,
                part_k=PART_K, part_emb=PART_EMB)


def train_model(key, epochs=None, verbose_every=None):
    epochs = NUM_EPOCHS if epochs is None else epochs
    verbose_every = max(1, epochs // 12) if verbose_every is None else verbose_every
    meta = _train_meta()
    path = os.path.join(CKPT_DIR, f"model_{safe_key(key)}.pt")
    model = build_model(key).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    sched = (torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs,
                                                        eta_min=LEARNING_RATE / 100)
             if LR_SCHEDULE == "cosine" else None)
    start, history = 0, []
    if RESUME and not FORCE_RETRAIN and os.path.exists(path):
        try:
            z = torch.load(path, map_location=DEV, weights_only=False)
            if z.get("meta") == meta:
                model.load_state_dict(z["model"]); opt.load_state_dict(z["opt"])
                if sched is not None and z.get("sched") is not None:
                    sched.load_state_dict(z["sched"])
                start, history = z["epoch"] + 1, z["history"]
                print(f"  [{key}] resumed from epoch {start}/{epochs}")
            else:
                print(f"  [{key}] checkpoint config differs from §1 -> retraining from scratch")
        except Exception as e:
            print(f"  [{key}] could not load checkpoint ({type(e).__name__}) -> retraining")
    if start >= epochs:
        print(f"  [{key}] already trained for {epochs} epochs (loaded from checkpoint).")
        model.eval(); hist = pd.DataFrame(history)
        hist.to_csv(os.path.join(HIST_DIR, f"history_{safe_key(key)}.csv"), index=False)
        return model, hist

    t_start, stopped = time.time(), None
    for ep in range(start, epochs):
        model.train(); tot, cnt = 0.0, 0
        for si, b in enumerate(epoch_batches(NUM_TRAIN_SHAPES, BATCH_SIZE, ep)):
            G0 = DATA["train"]["G0"][b].to(DEV); C0 = DATA["train"]["C0"][b].to(DEV)
            S = DATA["train"]["S"][b].to(DEV); ctx = ctx_for("train", b)
            t, noise = draw_t_noise(len(b), ep, si)
            loss = F.mse_loss(_fwd(model, key, DIF.q_sample(C0, t, noise), t, G0, C0, S, ctx), noise)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(b); cnt += len(b)
        if sched is not None:
            sched.step()
        # the frozen-bank pass is expensive for the LOCAL models; VAL_EVERY coarsens the curve
        # without changing the final number (the last epoch is always evaluated)
        do_val = (ep % VAL_EVERY == 0) or (ep == epochs - 1)
        vl = val_loss(model, key) if do_val else float("nan")
        history.append(dict(epoch=ep, train_loss=tot / cnt, val_loss=vl,
                            lr=opt.param_groups[0]["lr"], seconds=time.time() - t_start))
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(),
                        sched=None if sched is None else sched.state_dict(),
                        epoch=ep, history=history, meta=meta), path)
        if ep == start:
            per = time.time() - t_start        # epoch 0 always includes a val pass
            eta = (epochs - start) * (per * (1.0 / max(VAL_EVERY, 1)) + per * 0.6) / 60
            print(f"  [{key}] {per:.1f}s for the first epoch (incl. val) -> ETA ~{eta:.1f} min",
                  flush=True)
        if ep % verbose_every == 0 or ep == epochs - 1:
            vs = f"{vl:.4f}" if vl == vl else "  --  "
            print(f"  [{key}] ep {ep:4d}/{epochs}  train {tot/cnt:.4f}  val {vs}", flush=True)
        if TIME_BUDGET_MIN_PER_MODEL and (time.time() - t_start) / 60 > TIME_BUDGET_MIN_PER_MODEL:
            stopped = ep; print(f"  [{key}] time budget reached at epoch {ep}; stopping early")
            break
    hist = pd.DataFrame(history)
    hist.to_csv(os.path.join(HIST_DIR, f"history_{safe_key(key)}.csv"), index=False)
    model.eval()
    return model, hist


MODELS, HISTORY = {}, {}


def _guard():
    if not RUN_FULL_EXPERIMENT:
        print("RUN_FULL_EXPERIMENT is False -> skipping training.")
        print("Run §15 first; if it prints SANITY CHECK PASSED, set RUN_FULL_EXPERIMENT = True "
              "in §1, re-run §1, then come back.")
        return False
    if SANITY_PASSED is False:
        print("WARNING: §15 reported SANITY CHECK FAILED and you enabled RUN_FULL_EXPERIMENT "
              "anyway. Proceeding, but treat every number below as suspect.")
    elif SANITY_PASSED is None:
        print("NOTE: §15 was not run in this session, so the pipeline is unverified.")
    return True


print("training utilities ready. RUN_FULL_EXPERIMENT =", RUN_FULL_EXPERIMENT)
if RUN_SANITY_CHECK and SANITY is not None:
    # extrapolate the full run from the sanity timings actually measured on THIS machine
    _steps_ep = int(np.ceil(NUM_TRAIN_SHAPES / BATCH_SIZE))
    _est = {}
    for k in TRAIN_MODELS:
        _ms = SANITY[k]["losses"].shape[0]
        _per_step = SANITY_SECONDS[k] / _ms * (BATCH_SIZE / max(SANITY_CLOUDS, 1))
        _val = _per_step * (NUM_VAL_SHAPES * VAL_BANK_REPEATS / BATCH_SIZE) / 3.0 / max(VAL_EVERY, 1)
        _est[k] = (_per_step * _steps_ep + _val) * NUM_EPOCHS / 60
    print(f"\nestimated training time at {NUM_EPOCHS} epochs, VAL_EVERY={VAL_EVERY} "
          f"(from this session's §15 timings):")
    for k in TRAIN_MODELS:
        print(f"  {k:8s} ~{_est[k]:5.1f} min")
    print(f"  {'TOTAL':8s} ~{sum(_est.values()):5.1f} min"
          f"   (+ ~10 min for evaluation, figures and the report)")
    print("  To cut it: raise VAL_EVERY, drop models from TRAIN_MODELS (the LOCAL ones dominate),")
    print("  or lower NUM_EPOCHS. C / C+Gloc / C+GP is the minimum set that answers Question B.")

In [ ]:
if _guard():
    for _k in TRAIN_MODELS:
        t0 = time.time()
        MODELS[_k], HISTORY[_k] = train_model(_k)
        print(f"  {_k} trained in {(time.time()-t0)/60:.1f} min\n", flush=True)

MODELS_READY = all(k in MODELS for k in TRAIN_MODELS)
print("MODELS_READY =", MODELS_READY)
if MODELS_READY:
    fig, ax = plt.subplots(figsize=(7, 3.4))
    for k in TRAIN_MODELS:
        h = HISTORY[k]
        hv = h.dropna(subset=["val_loss"])          # val is sampled every VAL_EVERY epochs
        ax.plot(hv.epoch, hv.val_loss, lw=1.6, marker="o", ms=2.5, label=f"{k} val")
        ax.plot(h.epoch, h.train_loss, lw=0.8, alpha=.4)
    ax.set_xlabel("epoch"); ax.set_ylabel(r"$\epsilon$-MSE"); ax.legend(fontsize=7)
    ax.set_title("colour denoising — validation loss (thin = train)", fontsize=10)
    fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "16_training_curves.png"))
    plt.show(); plt.close(fig)
    pd.concat([HISTORY[k].assign(model=k) for k in MODELS]).to_csv(
        os.path.join(HIST_DIR, "history_all.csv"), index=False)
    print("\nfinal losses (train / val) — a conditioned model with LOWER train but HIGHER val than "
          "its\nbaseline is overfitting the extra channels, not exploiting them:")
    for k in TRAIN_MODELS:
        h = HISTORY[k]
        tr = float(h.train_loss.iloc[-1])
        vl = float(h.dropna(subset=["val_loss"]).val_loss.iloc[-1])
        print(f"  {k:7s} train {tr:.4f}   val {vl:.4f}")

## 17 · Quantitative evaluation

For each $(t, r)$ the noise $\epsilon$ is drawn from a generator seeded by $(t, r)$ alone, $C_t$ is
built once, and **every model in the ladder denoises that identical tensor** — not just the two
members of a pair. Combined with the shared kNN cache, the only thing that differs between models
is the model.

$\hat C_0$ is recovered with the §11 formula (clamped to $[-1,1]$) and scored with the full §14
suite. The **overall** row is the equal-weight mean over the `EVAL_TIMESTEPS` grid; a separate
uniform-$t$ number comes from the frozen validation bank.

Both the ratio-of-means improvement (the formula specified for this study) and the **mean of the
per-timestep improvements** are reported. They can differ a lot: $\epsilon$-MSE spans two orders of
magnitude across $t$, so the ratio-of-means is dominated by the low-$t$ end. Where they disagree the
per-timestep table in §18 is the informative one.

In [ ]:
EVAL_SEED = SEED + 90210


def df_to_md(df, fmt="{:.4g}", index=False):
    d = df.reset_index() if index else df
    cols = list(d.columns)
    def f(v):
        if isinstance(v, (float, np.floating)):
            return "" if (v != v) else fmt.format(v)
        return str(v)
    out = ["| " + " | ".join(map(str, cols)) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    for _, r in d.iterrows():
        out.append("| " + " | ".join(f(r[c]) for c in cols) + " |")
    return "\n".join(out)


def rel_improve(b, c, metric):
    '''100*(L_base - L_cond)/L_base for lower-is-better; sign-flipped for higher-is-better.'''
    if b is None or c is None or b != b or c != c or abs(b) < 1e-15:
        return float("nan")
    d = direction(metric)
    if d == "target1":
        return float("nan")
    return 100.0 * ((c - b) / abs(b) if d == "higher" else (b - c) / abs(b))


@torch.no_grad()
def make_xt(t_val, r):
    '''The corrupted colour tensor EVERY model will see. Depends only on (t, r).'''
    C0 = DATA["val"]["C0"].to(DEV)
    g = torch.Generator().manual_seed(EVAL_SEED + t_val * 1009 + r * 31)
    noise = torch.randn(NUM_VAL_SHAPES, NUM_POINTS, 3, generator=g).to(DEV)
    t = torch.full((NUM_VAL_SHAPES,), t_val, dtype=torch.long, device=DEV)
    return C0, t, noise, DIF.q_sample(C0, t, noise)


@torch.no_grad()
def eval_at_t(t_val, repeats):
    rows, prows = [], []
    G0 = DATA["val"]["G0"].to(DEV); S = DATA["val"]["S"].to(DEV)
    ctx = ctx_for("val", np.arange(NUM_VAL_SHAPES))
    for r in range(repeats):
        C0, t, noise, x_t = make_xt(t_val, r)
        for key in TRAIN_MODELS:
            eps = _fwd(MODELS[key], key, x_t, t, G0, C0, S, ctx)
            per_shape_eps = ((eps - noise) ** 2).mean(dim=(1, 2)).cpu().numpy()
            c0h = DIF.x0_from_eps(x_t, t, eps).cpu().numpy()
            for i in range(NUM_VAL_SHAPES):
                rgb_pred = np.clip((c0h[i] + 1) / 2, 0, 1)
                m = colour_metrics(VAL_CTX[i], rgb_pred)
                rows.append(dict(model=key, t=t_val, repeat=r, shape=i,
                                 shape_id=DATA["val"]["ids"][i],
                                 eps_mse=float(per_shape_eps[i]), **m))
                for p, v in VAL_CTX[i].per_part_dE00(srgb_to_lab(rgb_pred)).items():
                    prows.append(dict(model=key, t=t_val, repeat=r, shape=i, part=p,
                                      name=part_name(p),
                                      n_pts=VAL_CTX[i].part_n[p], deltaE00=v))
    return rows, prows


if MODELS_READY:
    t0 = time.time(); rows, prows = [], []
    for tv in EVAL_TIMESTEPS:
        a, b = eval_at_t(tv, EVAL_REPEATS)
        rows += a; prows += b
        print(f"  t={tv:4d} done ({time.time()-t0:5.1f}s)", flush=True)
    EVAL_DF = pd.DataFrame(rows)
    PART_DF = pd.DataFrame(prows)
    EVAL_DF.to_csv(os.path.join(TAB_DIR, "eval_raw.csv"), index=False)
    PART_DF.to_csv(os.path.join(TAB_DIR, "eval_per_part.csv"), index=False)
    print(f"\n{len(EVAL_DF)} evaluation rows and {len(PART_DF)} per-part rows "
          f"({NUM_VAL_SHAPES} shapes x {len(EVAL_TIMESTEPS)} t x {EVAL_REPEATS} repeats x "
          f"{len(TRAIN_MODELS)} models)")
else:
    EVAL_DF = PART_DF = None
    print("Models not trained — §17 onwards is skipped. Run §15, then enable RUN_FULL_EXPERIMENT.")

In [ ]:
if EVAL_DF is not None:
    UNIFORM = {k: val_loss(MODELS[k], k) for k in TRAIN_MODELS}
    print("uniform-t eps-MSE on the frozen validation bank (trivial predictor = 1.0):")
    for k in TRAIN_MODELS:
        print(f"  {k:8s} {UNIFORM[k]:.5f}")

    _mcols = [c for c in EVAL_DF.columns if c not in ("model", "t", "repeat", "shape", "shape_id")]
    PER_SHAPE = EVAL_DF.groupby(["model", "shape", "shape_id"], as_index=False)[_mcols].mean()
    BY_MODEL_T = EVAL_DF.groupby(["model", "t"], as_index=False)[_mcols].mean()
    BY_MODEL = EVAL_DF.groupby(["model"], as_index=False)[_mcols].mean()
    BY_MODEL["eps_mse_uniform_t"] = BY_MODEL["model"].map(UNIFORM)
    for nm, d in (("per_shape", PER_SHAPE), ("summary_by_model_t", BY_MODEL_T),
                  ("summary_by_model", BY_MODEL)):
        d.to_csv(os.path.join(TAB_DIR, f"{nm}.csv"), index=False)

    _order = {k: i for i, k in enumerate(TRAIN_MODELS)}
    _bm = BY_MODEL.set_index("model").loc[TRAIN_MODELS]
    print("\n--- colour suite, averaged over the evaluated t grid ---")
    print(_bm[HEADLINE_METRICS].to_string(float_format=lambda v: f"{v:.5f}"))
    print("\n--- part-aware + boundary metrics ---")
    print(_bm[PART_METRICS + ["part_consistency_ratio"]].to_string(
        float_format=lambda v: f"{v:.5f}"))

In [ ]:
if EVAL_DF is not None:
    try:
        from scipy.stats import wilcoxon, pearsonr, spearmanr
    except Exception:
        wilcoxon = pearsonr = spearmanr = None

    def improvement_by_t(base, cond, metric):
        d = BY_MODEL_T
        b = d[d.model == base].set_index("t")[metric].sort_index()
        c = d[d.model == cond].set_index("t")[metric].sort_index()
        return np.array([rel_improve(x, y, metric) for x, y in zip(b.values, c.values)])

    def paired_stats(base, cond, metric, tag=""):
        '''Paired over validation OBJECTS: every model saw identical noise on each one.'''
        b = PER_SHAPE[PER_SHAPE.model == base].set_index("shape")[metric].sort_index().values
        c = PER_SHAPE[PER_SHAPE.model == cond].set_index("shape")[metric].sort_index().values
        b, c = b.astype(float), c.astype(float)
        keep = np.isfinite(b) & np.isfinite(c)
        b, c = b[keep], c[keep]
        base_row = dict(question=tag, baseline=base, conditioned=cond, metric=metric,
                        direction=direction(metric))
        if len(b) == 0:
            return dict(base_row, baseline_value=np.nan, conditioned_value=np.nan,
                        improvement_pct=np.nan, improvement_pct_mean_over_t=np.nan,
                        ci_lo=np.nan, ci_hi=np.nan, win_rate=np.nan, n_shapes=0, wilcoxon_p=np.nan)
        d = (c - b) if direction(metric) == "higher" else (b - c)
        rng = np.random.default_rng(0)
        idx = rng.integers(0, len(d), (4000, len(d)))
        with np.errstate(divide="ignore", invalid="ignore"):
            boot = 100 * d[idx].mean(1) / np.abs(b[idx].mean(1))
        lo, hi = (np.percentile(boot[np.isfinite(boot)], [2.5, 97.5])
                  if np.isfinite(boot).any() else (np.nan, np.nan))
        p = np.nan
        if wilcoxon is not None and len(d) >= 6 and np.any(d != 0):
            try:
                p = float(wilcoxon(b, c).pvalue)
            except Exception:
                pass
        imp_t = improvement_by_t(base, cond, metric)
        return dict(base_row, baseline_value=float(b.mean()), conditioned_value=float(c.mean()),
                    improvement_pct=rel_improve(float(b.mean()), float(c.mean()), metric),
                    improvement_pct_mean_over_t=float(np.nanmean(imp_t)),
                    ci_lo=float(lo), ci_hi=float(hi), win_rate=float((d > 0).mean()),
                    n_shapes=int(len(d)), wilcoxon_p=p)

    PAIRED = pd.DataFrame([paired_stats(b, c, m, tag)
                           for tag, b, c, _q in COMPARISONS
                           for m in ALL_METRICS])
    PAIRED.to_csv(os.path.join(TAB_DIR, "paired_stats.csv"), index=False)

    def status(r):
        if r is None or len(r) == 0:
            return "unresolved"
        r = r.iloc[0] if hasattr(r, "iloc") else r
        if r.ci_lo != r.ci_lo:
            return "unresolved"
        if r.ci_lo > 0:
            return "helps"
        if r.ci_hi < 0:
            return "hurts"
        return "unresolved"

    for tag, b, c, q in COMPARISONS:
        print(f"\n{'='*88}\nQuestion {tag}: {b} vs {c}\n  {q}\n{'='*88}")
        sub = PAIRED[(PAIRED.question == tag) & (PAIRED.metric.isin(HEADLINE_METRICS))]
        print(sub[["metric", "direction", "baseline_value", "conditioned_value", "improvement_pct",
                   "improvement_pct_mean_over_t", "ci_lo", "ci_hi", "win_rate", "wilcoxon_p"]]
              .to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    print("\nwin_rate = fraction of validation OBJECTS on which the conditioned model wins.")
    print("ci spanning 0 => not resolved at this sample size. ci_hi < 0 => resolved as HARMFUL.")

In [ ]:
if EVAL_DF is not None:
    LEAK_TRAINED = leakage_table(MODELS)
    LEAK_TRAINED.to_csv(os.path.join(TAB_DIR, "leakage_test_trained.csv"), index=False)
    print("leakage re-test on the TRAINED weights (same function as §13c):\n")
    print(LEAK_TRAINED.to_string(index=False))
    assert (LEAK_TRAINED.verdict == "PASS").all(), "leakage detected after training"
    print("\nA trained model cannot have learned to exploit an input it never receives; this "
          "confirms\nthe wiring survived training and checkpoint round-trips.")

## 18 · Evaluation across diffusion timesteps

The previous notebook found that geometry conditioning *hurts* when colour is nearly clean and
*helps* once it is mostly destroyed — the crossover sits near $t\approx250$. The interesting
question here is whether part-awareness follows the same curve or a different one.

A result worth watching for:

> Part-aware conditioning becomes increasingly helpful as the original colour signal is destroyed.

That would matter, because heavy corruption is the regime a completion model actually operates in.

In [ ]:
def timestep_table(base, cond, metric):
    d = BY_MODEL_T
    b = d[d.model == base].set_index("t")[metric].sort_index()
    c = d[d.model == cond].set_index("t")[metric].sort_index()
    out = pd.DataFrame({"t": b.index.values, base: b.values, cond: c.values})
    out["improvement_%"] = [rel_improve(x, y, metric) for x, y in zip(b.values, c.values)]
    return out


if EVAL_DF is not None:
    TS_METRICS = ["eps_mse", "deltaE00", "part_macro_dE00", "part_swd_macro",
                  "boundary_dE00", "interior_dE00", "region_dE", "colour_edge_iou"]
    TS_TABLES = {}
    blocks = []
    for tag, b, c, _q in COMPARISONS:
        for m in TS_METRICS:
            TS_TABLES[(tag, m)] = timestep_table(b, c, m)
            blocks.append(TS_TABLES[(tag, m)].rename(columns={b: "baseline", c: "conditioned"})
                          .assign(question=tag, baseline_model=b, conditioned_model=c, metric=m))
    pd.concat(blocks).to_csv(os.path.join(TAB_DIR, "timestep_comparisons.csv"), index=False)

    for tag, b, c, _q in COMPARISONS:
        for m in ("eps_mse", "deltaE00", "part_macro_dE00"):
            print(f"\n--- Q{tag}  {b} -> {c} : {m} ({arrow(m)}) ---")
            print(TS_TABLES[(tag, m)].to_string(index=False, float_format=lambda v: f"{v:.5f}"))

In [ ]:
if EVAL_DF is not None:
    n = len(TS_METRICS)
    for tag, b, c, q in COMPARISONS:
        fig, axes = plt.subplots(2, n, figsize=(2.3 * n, 5.0), squeeze=False)
        for j, m in enumerate(TS_METRICS):
            tt = TS_TABLES[(tag, m)]
            ax = axes[0][j]
            ax.plot(tt.t, tt[b], "o-", lw=1.3, label=b); ax.plot(tt.t, tt[c], "s--", lw=1.3, label=c)
            ax.set_title(f"{m} {arrow(m)}", fontsize=7.5); ax.set_xlabel("t")
            if j == 0: ax.legend(fontsize=6)
            a2 = axes[1][j]
            a2.bar(tt.t, tt["improvement_%"], width=40,
                   color=["#2f9d66" if v > 0 else "#e0574c" for v in tt["improvement_%"]])
            a2.axhline(0, color="k", lw=.8); a2.set_xlabel("t")
            a2.set_title("improvement %", fontsize=7.5)
        fig.suptitle(f"Question {tag} — {b} vs {c}", fontsize=10)
        fig.tight_layout()
        fig.savefig(os.path.join(FIG_DIR, f"18_timestep_Q{tag}.png"), bbox_inches="tight")
        plt.show(); plt.close(fig)

## 19 · Part-level and boundary analysis

Three things the aggregate numbers cannot show:

1. **Which parts** benefit. A gain concentrated in the fuselage and absent from the engines means
   something different from a uniform gain.
2. **Interiors versus boundaries.** Hard same-part masking should make part interiors easier and
   risks making transitions worse. If `C+GP` improves `interior_dE00` while degrading
   `boundary_dE00`, that is the signature of cross-part context being genuinely useful at the seam —
   and it is the main way this idea can fail while still looking good on average.
3. **Whether the benefit tracks the assumption.** §20 handles that one.

In [ ]:
if EVAL_DF is not None:
    pp = (PART_DF.groupby(["model", "part", "name"], as_index=False)
          .agg(deltaE00=("deltaE00", "mean"), n_pts=("n_pts", "median"),
               objects=("shape", "nunique")))
    PART_TABLE = pp.pivot_table(index=["part", "name", "n_pts", "objects"],
                                columns="model", values="deltaE00").reset_index()
    cols = [c for c in TRAIN_MODELS if c in PART_TABLE.columns]
    if "C+Gloc" in cols and "C+GP" in cols:
        PART_TABLE["improvement_%_B"] = 100 * (PART_TABLE["C+Gloc"] - PART_TABLE["C+GP"]) \
            / PART_TABLE["C+Gloc"]
    PART_TABLE.to_csv(os.path.join(TAB_DIR, "per_part_table.csv"), index=False)
    print("per-part ΔE00 (mean over objects, timesteps and noise draws)\n")
    print(PART_TABLE.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

    # boundary vs interior, per model
    BI = BY_MODEL.set_index("model").loc[TRAIN_MODELS,
                                         ["boundary_dE00", "interior_dE00",
                                          "boundary_interior_ratio", "deltaE00"]]
    BI.to_csv(os.path.join(TAB_DIR, "boundary_interior.csv"))
    print("\nboundary vs interior ΔE00 (↓ both)\n")
    print(BI.to_string(float_format=lambda v: f"{v:.4f}"))
    print("\n  boundary_interior_ratio > 1 means transitions are harder than interiors (expected).")
    print("  The risk to watch: C+GP lowering interior_dE00 while RAISING boundary_dE00.")

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
    x = np.arange(len(TRAIN_MODELS))
    axes[0].bar(x - .2, BI.boundary_dE00.values, .4, label="boundary")
    axes[0].bar(x + .2, BI.interior_dE00.values, .4, label="interior")
    axes[0].set_xticks(x); axes[0].set_xticklabels(TRAIN_MODELS, rotation=20, fontsize=7)
    axes[0].set_ylabel("ΔE00 ↓"); axes[0].legend(fontsize=7)
    axes[0].set_title("boundary vs interior error", fontsize=9)
    if len(cols) >= 2:
        w = 0.8 / len(cols)
        for j, k in enumerate(cols):
            axes[1].bar(np.arange(len(PART_TABLE)) + j * w - 0.4, PART_TABLE[k].values, w, label=k)
        axes[1].set_xticks(np.arange(len(PART_TABLE)))
        axes[1].set_xticklabels(PART_TABLE["name"], fontsize=7)
        axes[1].set_ylabel("ΔE00 ↓"); axes[1].legend(fontsize=6)
        axes[1].set_title("per-part error", fontsize=9)
    d = PER_SHAPE
    for k in TRAIN_MODELS:
        v = d[d.model == k].sort_values("shape")["part_macro_dE00"].values
        axes[2].plot(np.sort(v), np.linspace(0, 1, len(v)), lw=1.4, label=k)
    axes[2].set_xlabel("part-macro ΔE00 ↓"); axes[2].set_ylabel("cumulative fraction of objects")
    axes[2].legend(fontsize=6); axes[2].set_title("distribution over objects", fontsize=9)
    fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "19_part_analysis.png"))
    plt.show(); plt.close(fig)

## 20 · Does the benefit track the assumption?

This is the analysis that connects the inductive bias to the observed effect rather than just
reporting an aggregate. For every validation object define

$$\mathrm{Gain}_i=\Delta E_{00}^{\,C+Gloc}(i)-\Delta E_{00}^{\,C+GP}(i),$$

positive when part-aware masking helped *that object*, and correlate it with how coherent that
object's parts actually are ($R_{micro}$ from §9 — **lower means more coherent**).

If part-awareness works *for the reason claimed*, objects with more colour-coherent parts should
gain more, i.e. Gain should be **negatively** correlated with $R$. A positive aggregate gain with
**no** such correlation would be a warning: the improvement would be coming from somewhere other
than the stated mechanism, and any story built on the mechanism would be unsupported.

Pearson (linear) and Spearman (rank) are both reported — with 40 objects a single outlier can drive
Pearson, so disagreement between them is itself informative.

In [ ]:
GAIN = CORR = None
if EVAL_DF is not None and "C+Gloc" in TRAIN_MODELS and "C+GP" in TRAIN_MODELS:
    base_k, cond_k = "C+Gloc", "C+GP"
    rows = []
    for i in range(NUM_VAL_SHAPES):
        # NB: PER_SHAPE["shape"], never PER_SHAPE.shape — the latter is the DataFrame's dimensions
        rb = PER_SHAPE[(PER_SHAPE.model == base_k) & (PER_SHAPE["shape"] == i)]
        rc = PER_SHAPE[(PER_SHAPE.model == cond_k) & (PER_SHAPE["shape"] == i)]
        if len(rb) == 0 or len(rc) == 0:
            continue
        rows.append(dict(shape=i, shape_id=DATA["val"]["ids"][i],
                         R_micro=VAL_CTX[i].coherence_R,
                         boundary_frac=float(VAL_CTX[i].boundary.mean()),
                         n_parts=len(VAL_CTX[i].part_ids),
                         gain_dE00=float(rb.deltaE00.values[0] - rc.deltaE00.values[0]),
                         gain_part_macro=float(rb.part_macro_dE00.values[0]
                                               - rc.part_macro_dE00.values[0]),
                         gain_eps=float(rb.eps_mse.values[0] - rc.eps_mse.values[0])))
    GAIN = pd.DataFrame(rows)
    if len(GAIN) == 0 or not np.isfinite(GAIN.R_micro.values).any():
        print("no usable per-object coherence values — skipping the correlation analysis")
        GAIN = None

if GAIN is not None:
    GAIN = GAIN.dropna(subset=["R_micro"])
    GAIN.to_csv(os.path.join(TAB_DIR, "coherence_vs_gain.csv"), index=False)

    def corr(x, y):
        m = np.isfinite(x) & np.isfinite(y)
        if m.sum() < 5 or pearsonr is None:
            return (float("nan"),) * 4
        pr, pp = pearsonr(x[m], y[m]); sr, sp = spearmanr(x[m], y[m])
        return float(pr), float(pp), float(sr), float(sp)

    rows = []
    for gm in ("gain_dE00", "gain_part_macro", "gain_eps"):
        pr, pp, sr, sp = corr(GAIN.R_micro.values, GAIN[gm].values)
        rows.append(dict(gain_metric=gm, vs="R_micro (lower = more coherent)",
                         pearson_r=pr, pearson_p=pp, spearman_r=sr, spearman_p=sp,
                         n=int(np.isfinite(GAIN[gm].values).sum())))
    CORR = pd.DataFrame(rows)
    CORR.to_csv(os.path.join(TAB_DIR, "coherence_gain_correlation.csv"), index=False)
    print(f"Gain_i = dE00({base_k}) - dE00({cond_k}) — positive means part-masking helped object i\n")
    print(f"  objects with a positive gain: {100*np.mean(GAIN.gain_dE00 > 0):.0f}%   "
          f"mean gain {GAIN.gain_dE00.mean():+.4f} dE00")
    print("\ncorrelation with within-part colour coherence:\n")
    print(CORR.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    print("\n  The MECHANISM predicts a NEGATIVE correlation with R_micro (more coherent parts ->\n"
          "  bigger gain). A positive aggregate gain with r ~ 0 means the improvement is not coming\n"
          "  from the stated mechanism.")

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
    for ax, gm, ttl in zip(axes, ["gain_dE00", "gain_part_macro", "gain_eps"],
                           ["dE00 gain", "part-macro dE00 gain", "eps-MSE gain"]):
        ax.scatter(GAIN.R_micro, GAIN[gm], s=22, alpha=.8)
        m = np.isfinite(GAIN.R_micro.values) & np.isfinite(GAIN[gm].values)
        if m.sum() >= 2:
            z = np.polyfit(GAIN.R_micro.values[m], GAIN[gm].values[m], 1)
            xs = np.linspace(GAIN.R_micro.min(), GAIN.R_micro.max(), 20)
            ax.plot(xs, np.polyval(z, xs), "r--", lw=1.2)
        ax.axhline(0, color="k", lw=.8)
        ax.set_xlabel("$R_{micro}$  (lower = parts more coherent)"); ax.set_ylabel(ttl)
        ax.set_title(f"{ttl} vs coherence", fontsize=9)
    fig.suptitle("Does part-awareness help most where parts really are colour-coherent?",
                 fontsize=10)
    fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "20_coherence_vs_gain.png"))
    plt.show(); plt.close(fig)
else:
    print("skipped (needs both C+Gloc and C+GP trained)")

## 21 · Visual comparisons

Same validation objects, same camera, same noise draw (evaluation repeat 0) for every model — and
crucially a **shared ΔE00 colour scale** across models so the error maps are directly comparable.
Each row shows ground truth, the semantic part map, the noisy input, and every model's
reconstruction, followed by the matching error maps.

In [ ]:
@torch.no_grad()
def reconstruct(t_val, r=0):
    '''x0-hat for EVERY model on all val shapes, from evaluation repeat r's exact noise.'''
    G0 = DATA["val"]["G0"].to(DEV); S = DATA["val"]["S"].to(DEV)
    ctx = ctx_for("val", np.arange(NUM_VAL_SHAPES))
    C0, t, noise, x_t = make_xt(t_val, r)
    out = {"x_t": x_t.cpu().numpy()}
    for key in TRAIN_MODELS:
        eps = _fwd(MODELS[key], key, x_t, t, G0, C0, S, ctx)
        out[key] = DIF.x0_from_eps(x_t, t, eps).cpu().numpy()
    return out


def to_rgb(a):
    return np.clip((a + 1) / 2, 0, 1)


if EVAL_DF is not None:
    if VIS_T not in EVAL_TIMESTEPS:
        print(f"NOTE: VIS_T={VIS_T} is not in EVAL_TIMESTEPS; these figures show a noise level with "
              f"no row in the tables.")
    VIS_IDX = list(range(min(N_VIS_SHAPES, NUM_VAL_SHAPES)))
    REC = reconstruct(VIS_T, 0)

    ncol = 3 + len(TRAIN_MODELS)
    panels = []
    for i in VIS_IDX:
        xyz = DATA["val"]["xyz"][i]
        panels += [(f"[{i}] GT colour", xyz, DATA["val"]["rgb"][i]),
                   (f"[{i}] parts", xyz, part_rgb(DATA["val"]["part"][i])),
                   (f"[{i}] noisy $C_t$ t={VIS_T}", xyz, to_rgb(REC["x_t"][i]))]
        panels += [(f"[{i}] {k}", xyz, to_rgb(REC[k][i])) for k in TRAIN_MODELS]
    grid3d(panels, ncols=ncol, figsize_per=2.2,
           suptitle=f"colour reconstruction at t={VIS_T} (geometry fixed at $G_0$; noisy colours "
                    f"clipped for display)",
           path=os.path.join(FIG_DIR, f"21_reconstructions_t{VIS_T}.png"))

In [ ]:
if EVAL_DF is not None:
    # ---- ΔE00 error maps on a SHARED colour scale ----
    for i in VIS_IDX:
        xyz, gt = DATA["val"]["xyz"][i], DATA["val"]["rgb"][i]
        errs = {k: delta_e00(gt, to_rgb(REC[k][i])) for k in TRAIN_MODELS}
        vmax = float(np.percentile(np.concatenate(list(errs.values())), 97))
        n = len(TRAIN_MODELS) + 2
        fig = plt.figure(figsize=(2.4 * n, 2.9))
        ax = fig.add_subplot(1, n, 1, projection="3d"); plot_cloud(ax, xyz, gt, "GT colour")
        ax = fig.add_subplot(1, n, 2, projection="3d")
        plot_cloud(ax, xyz, part_rgb(DATA["val"]["part"][i]), "parts")
        for j, k in enumerate(TRAIN_MODELS):
            ax = fig.add_subplot(1, n, j + 3, projection="3d")
            sc = plot_cloud(ax, xyz, None, f"{k}\nmean ΔE00={errs[k].mean():.1f}",
                            cmap_vals=errs[k], cmap="magma", vmin=0, vmax=vmax)
        fig.colorbar(sc, ax=fig.axes[-1], fraction=.03)
        fig.suptitle(f"object [{i}] {DATA['val']['ids'][i][:20]} — ΔE00 error at t={VIS_T} "
                     f"(shared scale 0–{vmax:.0f})", fontsize=9)
        fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, f"21_error_map_{i}.png"),
                                        bbox_inches="tight")
        plt.show(); plt.close(fig)

In [ ]:
if EVAL_DF is not None and "C+Gloc" in TRAIN_MODELS and "C+GP" in TRAIN_MODELS:
    # ---- boundary-focused comparison: where does masking help or hurt? ----
    for i in VIS_IDX[:2]:
        xyz, gt, part = DATA["val"]["xyz"][i], DATA["val"]["rgb"][i], DATA["val"]["part"][i]
        b = VAL_CTX[i].boundary
        e_base = delta_e00(gt, to_rgb(REC["C+Gloc"][i]))
        e_part = delta_e00(gt, to_rgb(REC["C+GP"][i]))
        diff = e_base - e_part                       # >0 where C+GP is better
        v = float(np.percentile(np.abs(diff), 97))
        fig = plt.figure(figsize=(12, 2.9))
        ax = fig.add_subplot(1, 4, 1, projection="3d")
        plot_cloud(ax, xyz, np.where(b[:, None], np.array([[.85, .2, .2]]),
                                     np.array([[.82, .82, .86]])),
                   f"semantic boundary ({100*b.mean():.0f}%)")
        ax = fig.add_subplot(1, 4, 2, projection="3d")
        sc = plot_cloud(ax, xyz, None, "C+Gloc ΔE00", cmap_vals=e_base, cmap="magma",
                        vmin=0, vmax=float(np.percentile(np.r_[e_base, e_part], 97)))
        ax = fig.add_subplot(1, 4, 3, projection="3d")
        sc2 = plot_cloud(ax, xyz, None, "C+GP ΔE00", cmap_vals=e_part, cmap="magma",
                         vmin=0, vmax=float(np.percentile(np.r_[e_base, e_part], 97)))
        fig.colorbar(sc2, ax=ax, fraction=.03)
        ax = fig.add_subplot(1, 4, 4, projection="3d")
        sc3 = plot_cloud(ax, xyz, None, "blue = C+GP better", cmap_vals=diff, cmap="coolwarm_r",
                         vmin=-v, vmax=v)
        fig.colorbar(sc3, ax=ax, fraction=.03)
        fig.suptitle(f"object [{i}] — does same-part masking help at the seams or only inside "
                     f"parts?  (t={VIS_T})", fontsize=9)
        fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, f"21_boundary_effect_{i}.png"),
                                        bbox_inches="tight")
        plt.show(); plt.close(fig)

        db = pd.DataFrame([dict(region="boundary", n=int(b.sum()),
                                C_Gloc=float(e_base[b].mean()), C_GP=float(e_part[b].mean())),
                           dict(region="interior", n=int((~b).sum()),
                                C_Gloc=float(e_base[~b].mean()), C_GP=float(e_part[~b].mean()))])
        db["improvement_%"] = 100 * (db.C_Gloc - db.C_GP) / db.C_Gloc
        print(f"object [{i}] ΔE00 by region at t={VIS_T}:")
        print(db.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

In [ ]:
if EVAL_DF is not None and SAVE_PLOTLY:
    def write_panels(path, panels):
        import plotly.graph_objects as go
        from plotly.subplots import make_subplots
        fig = make_subplots(rows=1, cols=len(panels), specs=[[{"type": "scene"}] * len(panels)],
                            subplot_titles=[t for t, _, _ in panels])
        for col, (_, xyz, rgb) in enumerate(panels, 1):
            c = ["rgb(%d,%d,%d)" % tuple((np.clip(v, 0, 1) * 255).astype(int)) for v in rgb]
            fig.add_trace(go.Scatter3d(x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2], mode="markers",
                                       marker=dict(size=1.8, color=c)), 1, col)
        fig.update_layout(height=520, showlegend=False, margin=dict(l=0, r=0, t=30, b=0))
        for i in range(1, len(panels) + 1):
            fig.layout["scene" if i == 1 else f"scene{i}"].aspectmode = "data"
        fig.write_html(path)

    try:
        i = VIS_IDX[0]; xyz = DATA["val"]["xyz"][i]
        write_panels(os.path.join(FIG_DIR, f"21_interactive_t{VIS_T}.html"),
                     [("GT", xyz, DATA["val"]["rgb"][i]),
                      ("parts", xyz, part_rgb(DATA["val"]["part"][i]))] +
                     [(k, xyz, to_rgb(REC[k][i])) for k in TRAIN_MODELS])
        print("interactive panel written to", FIG_DIR)
    except Exception as e:
        print("plotly export skipped:", type(e).__name__, e)

## 22 · Final result tables

The main paired table, plus per-part and per-object CSVs. Every comparison stays paired per object.

In [ ]:
if EVAL_DF is not None:
    MAIN = BY_MODEL.set_index("model").loc[TRAIN_MODELS, HEADLINE_METRICS].rename(
        columns={m: f"{m} {arrow(m)}" for m in HEADLINE_METRICS})
    MAIN.to_csv(os.path.join(TAB_DIR, "main_table.csv"))
    print("MAIN TABLE — colour denoising, averaged over the t grid and noise draws\n")
    print(MAIN.to_string(float_format=lambda v: f"{v:.4f}"))

    rows = []
    for tag, b, c, q in COMPARISONS:
        for m in HEADLINE_METRICS:
            r = PAIRED[(PAIRED.question == tag) & (PAIRED.metric == m)].iloc[0]
            rows.append(dict(Question=tag, Baseline=b, Conditioned=c, Metric=f"{m} {arrow(m)}",
                             Baseline_value=r.baseline_value, Conditioned_value=r.conditioned_value,
                             Improvement_pct=r.improvement_pct,
                             Improvement_mean_over_t=r.improvement_pct_mean_over_t,
                             CI=f"[{r.ci_lo:.1f}, {r.ci_hi:.1f}]", Win_rate=r.win_rate,
                             Wilcoxon_p=r.wilcoxon_p, Verdict=status(
                                 PAIRED[(PAIRED.question == tag) & (PAIRED.metric == m)])))
    HEADLINE = pd.DataFrame(rows)
    HEADLINE.to_csv(os.path.join(TAB_DIR, "headline.csv"), index=False)
    for tag, b, c, q in COMPARISONS:
        print(f"\n--- Question {tag}: {b} vs {c} ---")
        print(HEADLINE[HEADLINE.Question == tag].drop(columns=["Question", "Baseline",
                                                               "Conditioned"])
              .to_string(index=False, float_format=lambda v: f"{v:.4f}"))

    fig, axes = plt.subplots(1, len(COMPARISONS), figsize=(3.4 * len(COMPARISONS), 3.4),
                             squeeze=False)
    for ax, (tag, b, c, q) in zip(axes[0], COMPARISONS):
        v = [PAIRED[(PAIRED.question == tag) & (PAIRED.metric == m)].iloc[0].improvement_pct
             for m in HEADLINE_METRICS]
        ax.barh(range(len(HEADLINE_METRICS)), v,
                color=["#2f9d66" if (x == x and x > 0) else "#e0574c" for x in v])
        ax.set_yticks(range(len(HEADLINE_METRICS)))
        ax.set_yticklabels([f"{m} {arrow(m)}" for m in HEADLINE_METRICS], fontsize=6.5)
        ax.axvline(0, color="k", lw=.9); ax.set_xlabel("improvement %")
        ax.set_title(f"Q{tag}: {b} → {c}", fontsize=8)
    fig.suptitle("positive = the conditioned model helped (sign corrected per metric)", fontsize=9)
    fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "22_improvement_summary.png"))
    plt.show(); plt.close(fig)

## 23 · Automatic experimental report

Written from the CSVs this run produced. **Questions A, B and C are answered separately and never
merged into a single verdict.** For Question B the report applies the explicit evidence checklist —
a single improving metric is not treated as success — and if the answer is negative it works through
the candidate causes rather than asserting the hypothesis is false.

Output: `results/experiment_report.md` and `results/experiment_report.html`.

In [ ]:
def _load(name):
    p = os.path.join(TAB_DIR, name)
    return pd.read_csv(p) if os.path.exists(p) else None


def generate_report():
    par = _load("paired_stats.csv")
    if par is None:
        return ("# Oracle Part-Aware Colour Denoising\n\nNo results found in `%s`.\n\nRun §15, set "
                "`RUN_FULL_EXPERIMENT = True` in §1, then run §16 onward.\n" % TAB_DIR)
    coh, bnd = _load("h1_coherence_per_object.csv"), _load("h2_boundary_per_object.csv")
    ali, cat = _load("h2_edge_alignment.csv"), _load("h1_h2_category_summary.csv")
    pars, leak = _load("model_parameters.csv"), _load("leakage_test_trained.csv")
    san, main = _load("sanity_check.csv"), _load("main_table.csv")
    ptab, bi = _load("per_part_table.csv"), _load("boundary_interior.csv")
    corr, gain = _load("coherence_gain_correlation.csv"), _load("coherence_vs_gain.csv")

    def row(tag, metric):
        m = par[(par.question == tag) & (par.metric == metric)]
        return None if len(m) == 0 else m.iloc[0]

    def stat(r):
        if r is None or r.ci_lo != r.ci_lo:
            return "unresolved"
        return "helps" if r.ci_lo > 0 else ("hurts" if r.ci_hi < 0 else "unresolved")

    def verdict(r, nb, nc):
        if r is None:
            return "not measured"
        s = (f"{nc} {'improves on' if r.improvement_pct > 0 else 'is worse than'} {nb} by "
             f"**{abs(r.improvement_pct):.2f}%** ({r.baseline_value:.5f} → "
             f"{r.conditioned_value:.5f}), winning on {r.win_rate*100:.0f}% of "
             f"{int(r.n_shapes)} objects; 95% CI [{r.ci_lo:.1f}%, {r.ci_hi:.1f}%]")
        if r.wilcoxon_p == r.wilcoxon_p:
            s += f", Wilcoxon p={r.wilcoxon_p:.2g}"
        if abs(r.improvement_pct - r.improvement_pct_mean_over_t) > 2:
            s += (f" — note the mean of the per-timestep improvements is "
                  f"{r.improvement_pct_mean_over_t:+.2f}%, so the headline is dominated by the "
                  f"largest-magnitude timesteps")
        st = stat(r)
        return s + ("." if st != "unresolved" else
                    ". **The interval spans 0 — not resolved at this sample size.**")

    L = []; A = L.append
    A("# Oracle Part-Aware Colour Denoising — feasibility report\n")
    A(f"*Generated automatically from the run in `{RESULTS_DIR}`. Every number was produced by this "
      f"execution.*\n")
    if IS_SYNTHETIC:
        A("> **WARNING — SYNTHETIC FALLBACK DATA.** Pipeline smoke test only.\n")

    A("## 1. Objective and hypothesis\n")
    A("Tested here: **colour is more predictable when geometry→colour information is exchanged "
      "within the same semantic object part.** Geometry stays clean; only colour is diffused; part "
      "labels are **oracle** ground truth, so this measures the *upper bound* on what perfect part "
      "knowledge could buy, not a test-time method. Three questions are kept separate:\n\n"
      "- **A** — does clean geometry help colour denoising at all?\n"
      "- **B** — does semantic part structure add anything beyond geometry?\n"
      "- **C** — is any gain from *knowing* the part, or from *restricting information flow* to it?\n")

    A("## 2. Dataset, parts, sample sizes\n")
    A(f"- source `{DATA_ORIGIN}`, category **{CATEGORY}** (`{SYNSET}`)\n"
      f"- {NUM_TRAIN_SHAPES} train / {NUM_VAL_SHAPES} val objects, strictly disjoint\n"
      f"- {NUM_POINTS} points per cloud, subsampled from 8192 by `{SUBSAMPLE}`\n"
      f"- per-point ShapeNet-Part labels, already contiguous ids 0…{N_PARTS_MAX-1}"
      + (f" ({', '.join(PART_NAMES)})" if PART_NAMES else "") + "; no remapping applied\n"
      f"- `(G0_i, C0_i, s_i)` correspondence asserted after subsampling (§6)\n")

    A("## 3. Is the premise true in this data?\n")
    if coh is not None:
        rm, rn = float(coh.R_micro.median()), float(coh.R_micro_null.median())
        A(f"**H1 — within-part colour coherence.** $R_{{micro}}$ median **{rm:.3f}** "
          f"({100*float((coh.R_micro < 1).mean()):.0f}% of objects below 1). The random-partition "
          f"null of the same part sizes sits at {rn:.3f}, so semantic parts explain "
          f"**{(rn-rm)*100:+.1f} percentage points** of Lab dispersion beyond an arbitrary split.\n")
        if rm >= rn:
            A("> Semantic parts do **no better than an arbitrary partition**. The part-aware "
              "inductive bias is unmotivated for this category, and a null answer to Question B "
              "would be the expected outcome rather than a surprise.\n")
    if bnd is not None:
        A(f"**H2 — boundaries.** $D_{{inter}}/D_{{intra}}$ median "
          f"**{float(bnd.ratio.median()):.2f}** ({float(bnd.D_inter.median()):.2f} vs "
          f"{float(bnd.D_intra.median()):.2f} ΔE00), above 1 for "
          f"{100*float((bnd.ratio > 1).mean()):.0f}% of objects. Boundary points are "
          f"{100*float(bnd.boundary_frac.median()):.1f}% of the cloud — masking is "
          + ("a substantial intervention, not a gentle one" if float(bnd.boundary_frac.median()) > .2
             else "a fairly local intervention") + ".\n")
    if ali is not None:
        A(f"Semantic boundaries vs strong colour edges: recall "
          f"{float(ali.recall.median()):.2f} (chance {float(ali.chance_recall.median()):.2f}), "
          f"precision {float(ali.precision.median()):.2f} "
          f"(chance {float(ali.chance_precision.median()):.2f}).\n")
    if cat is not None:
        A(df_to_md(cat) + "\n")

    A("## 4. Diffusion and model ladder\n")
    A(f"DDPM forward process only, `T={T}`, `{BETA_SCHEDULE}` schedule; **colour is noised, "
      f"geometry is never noised**. $\\hat C_0$ recovered with the standard formula, clamped to "
      f"$[-1,1]$, identically for every model.\n")
    if pars is not None:
        A(df_to_md(pars[["model", "input", "geometry", "knn", "part_mode", "params"]]) + "\n")
    A(f"kNN candidates come from XYZ only (k={PART_K}); the part label only masks which candidates "
      f"may contribute. The mask keeps ~{100*float(GRAPH['val']['same'].float().mean()):.0f}% of "
      f"candidate edges. The self-edge fallback (no same-part neighbour at all) fires for "
      f"**{FALLBACK_FRAC*100:.3f}%** of validation points — it never borrows from another part.\n")
    if leak is not None:
        A("Leakage re-test on the **trained** weights — with $x_t$ and the topology fixed, each "
          "input is perturbed in turn:\n")
        A(df_to_md(leak) + "\n")
    A("Topology invariance (part labels do not change the kNN indices) and mask purity (no admitted "
      "neighbour crosses a part boundary) are asserted in §13c.\n")

    A("## 5. Training\n")
    A(f"AdamW lr {LEARNING_RATE} ({LR_SCHEDULE}), wd {WEIGHT_DECAY}, batch {BATCH_SIZE}, "
      f"{NUM_EPOCHS} epochs, seed {SEED}, device `{DEVICE}`. Plain noise-prediction MSE. All models "
      f"see identical batches, timesteps, noise and kNN topology.\n")
    if san is not None:
        ok = bool(san.passed.all())
        A(f"Tiny-set overfit check: **{'passed' if ok else 'FAILED'}** "
          f"{'for all models' if ok else 'for ' + str(san.loc[~san.passed,'model'].tolist())}.\n")
        A(df_to_md(san[["model", "loss_end", "ratio", "eps_t250", "x0_gain_t250", "passed"]]) + "\n")

    A("## 6. Main results\n")
    if main is not None:
        A(df_to_md(main, index=True) + "\n")

    for tag, b, c, q in COMPARISONS:
        A(f"### Question {tag} — {q}\n")
        A(f"`{b}` vs `{c}`:\n")
        for m in HEADLINE_METRICS:
            r = row(tag, m)
            if r is not None:
                A(f"- `{m}` {arrow(m)}: {verdict(r, b, c)}\n")

    A("## 7. Part-level and boundary breakdown\n")
    if ptab is not None:
        A(df_to_md(ptab) + "\n")
    if bi is not None:
        A("Boundary vs interior ΔE00 — the failure mode to look for is a model that lowers the "
          "interior while raising the boundary:\n")
        A(df_to_md(bi, index=True) + "\n")

    A("## 8. Does the benefit track the assumption?\n")
    if corr is not None and gain is not None and len(gain):
        cr = corr[corr.gain_metric == "gain_dE00"].iloc[0]
        A(f"Per object, $\\mathrm{{Gain}}_i=\\Delta E_{{00}}^{{C+Gloc}}-\\Delta E_{{00}}^{{C+GP}}$ "
          f"was positive for **{100*float((gain.gain_dE00 > 0).mean()):.0f}%** of objects "
          f"(mean {float(gain.gain_dE00.mean()):+.4f} ΔE00).\n")
        A(f"Correlation with $R_{{micro}}$ (lower = more coherent parts): Pearson "
          f"r={cr.pearson_r:+.3f} (p={cr.pearson_p:.3g}), Spearman ρ={cr.spearman_r:+.3f} "
          f"(p={cr.spearman_p:.3g}), n={int(cr.n)}.\n")
        if cr.pearson_r < -0.2 and cr.pearson_p < 0.1:
            A("The gain is **larger on objects whose parts really are colour-coherent** — the "
              "improvement behaves the way the stated mechanism predicts, which is stronger "
              "evidence than the aggregate number alone.\n")
        elif cr.pearson_r > 0.2 and cr.pearson_p < 0.1:
            A("The gain is larger where parts are *less* coherent — the **opposite** of the "
              "mechanism's prediction. Whatever is producing the aggregate difference, it is not "
              "intra-part colour coherence.\n")
        else:
            A("**No correlation between coherence and gain.** Any aggregate difference is therefore "
              "not traceable to the stated mechanism; treat a positive headline with caution.\n")

    A("## 9. Evidence checklist for Question B\n")
    A("A single improving metric is not sufficient. Applying the criteria set for this study:\n")
    checks, passed = [], 0
    if any(t == "B" for t, _, _, _ in COMPARISONS):
        def _imp(m):
            r = row("B", m)
            return (r.improvement_pct if r is not None else float("nan"))
        crit = [("ΔE00 improves", _imp("deltaE00") > 0),
                ("part-macro ΔE00 improves", _imp("part_macro_dE00") > 0),
                ("per-part distribution (part SWD) improves", _imp("part_swd_macro") > 0),
                ("region ΔE does not degrade", _imp("region_dE") > -1),
                ("local consistency does not collapse", _imp("part_consistency_absdiff") > -10),
                ("boundary performance does not degrade substantially",
                 _imp("boundary_dE00") > -2),
                ("interior improves", _imp("interior_dE00") > 0)]
        if corr is not None and len(corr):
            cr = corr[corr.gain_metric == "gain_dE00"].iloc[0]
            crit.append(("gain correlates with intra-part coherence", bool(cr.pearson_r < -0.2)))
        for nm, ok in crit:
            checks.append(dict(criterion=nm, met=bool(ok))); passed += int(bool(ok))
        A(df_to_md(pd.DataFrame(checks)) + "\n")
        A(f"**{passed} of {len(checks)} criteria met.**\n")

    A("## 10. Answers\n")
    for tag, b, c, q in COMPARISONS:
        r = row(tag, "deltaE00"); r2 = row(tag, "part_macro_dE00")
        s, s2 = stat(r), stat(r2)
        A(f"- **Question {tag}** (`{b}` → `{c}`): ΔE00 **{s}**, part-macro ΔE00 **{s2}**.\n")
    rb = row("B", "deltaE00")
    if rb is not None:
        if stat(rb) == "helps" and passed >= max(5, len(checks) - 2):
            A("\n**Question B — supported.** Under this controlled setup, with oracle part labels, "
              "restricting geometry→colour aggregation to same-part neighbourhoods measurably "
              "improves colour denoising, and the improvement survives the checklist. This does "
              "**not** show that a completion system with *predicted* parts would benefit — the "
              "labels here are oracle, and §11 lists what that leaves open.\n")
        elif stat(rb) == "hurts":
            A("\n**Question B — resolved negative.** Same-part masking measurably *hurt*. The most "
              "likely reasons, in order of how well this run supports them: cross-part context is "
              "genuinely useful (check the boundary row in §7); the mask discards too much of the "
              "neighbourhood (§4 reports how much); or part labels are too coarse to localise "
              "colour structure.\n")
        else:
            A("\n**Question B — not resolved.** This does not establish that part structure is "
              "useless. Candidate causes, to be ruled out in this order:\n"
              "1. **the premise is weak here** — §3's $R$ sits close to its random-partition null, "
              "so there is little intra-part coherence to exploit;\n"
              "2. **most objects are nearly single-coloured**, leaving no colour structure for any "
              "conditioning to recover;\n"
              "3. **part labels are too coarse** — 3–4 parts cannot localise within-part texture;\n"
              "4. **within-part texture is multimodal** (several flat materials inside one part), "
              "so a part is not a colour-homogeneous unit;\n"
              "5. **hard masking removes useful nearby context** — check whether `boundary_dE00` "
              "degraded while `interior_dE00` improved;\n"
              "6. **the unmasked model already extracts the same information** from geometry alone, "
              "making the part label redundant rather than wrong;\n"
              "7. **too few objects** for the effect size present.\n")

    A("## 11. Limitations\n")
    A(f"1. **Oracle part labels.** This is an upper bound. A predicted-segmentation version will "
      f"score lower — the previous part-colouring work in this repo found segmentation quality, "
      f"not the colouring rule, to be the binding constraint.\n"
      f"2. **Denoising, not completion.** No reverse sampler, no mask, no partial input.\n"
      f"3. **Clean geometry.** $G_0$ is a gift the completion task does not give.\n"
      f"4. **One category ({CATEGORY}), {NUM_TRAIN_SHAPES}+{NUM_VAL_SHAPES} objects, "
      f"{NUM_POINTS} points, single seed.**\n"
      f"5. **Hard masking is one design point.** Soft attention weighting over the same "
      f"neighbourhood was not tried and could behave differently at boundaries.\n"
      f"6. The $[-1,1]$ clamp dominates high-$t$ reconstruction metrics equally for all models; "
      f"`eps_mse` is the clamp-free number.\n")

    A("## 12. Recommended next experiment\n")
    if rb is not None and stat(rb) == "helps":
        A("Part-aware colour conditioning helps under oracle labels. The next step is **not** the "
          "full 6-D model — it is to find out how much of this survives contact with reality:\n\n"
          "1. **Replace oracle labels with predicted ones** (the repo already has a PointNet "
          "part-segmenter). Re-run Question B with predicted parts and report both rows. If the "
          "gain vanishes, the bottleneck is segmentation, not the idea.\n"
          "2. Then **soften the mask** — attention weighted by same-part membership rather than a "
          "hard cut — and check specifically whether the boundary metric recovers.\n\n"
          "Only after both would joint XYZ+RGB diffusion with global geometry reasoning and "
          "part-aware colour interaction be worth building.\n")
    else:
        A("Question B did not come back positive, so **do not build the joint 6-D model on the "
          "strength of the original plan**. Diagnose first, cheaply:\n\n"
          "1. **Test the premise harder** — §3's coherence ratio is the cheapest lever. Run the "
          "same analysis on the other two categories and pick whichever shows the largest gap "
          "between $R$ and its null. If none does, semantic parts are simply not the right unit "
          "for colour on this data.\n"
          "2. **Try finer units than semantic parts** — the FPS-Voronoi cells already used by "
          "`region_dE` give a geometry-only partition at any granularity, and would test whether "
          "the useful structure is *semantic* or merely *local*.\n"
          "3. Only if one of those shows a real effect is a joint model worth the cost.\n")
    A("\n---\n")
    A(f"*Config: seed {SEED}, {CATEGORY}, {NUM_POINTS} pts, T={T} ({BETA_SCHEDULE}), width "
      f"{WIDTH}x{N_BLOCKS}, k={PART_K}, {NUM_EPOCHS} epochs, eval t={EVAL_TIMESTEPS} x "
      f"{EVAL_REPEATS} draws, models {TRAIN_MODELS}.*\n")
    return "\n".join(L)


REPORT_MD = generate_report()
report_path = os.path.join(RESULTS_DIR, "experiment_report.md")
with open(report_path, "w") as f:
    f.write(REPORT_MD)
print("report written to", report_path, f"({len(REPORT_MD)} chars)\n")
print("=" * 78 + "\n")
print(REPORT_MD[:5000])
print("\n... (full text in the file above) ...")

In [ ]:
# ---- compact HTML twin of the report (no external markdown dependency) ----
import re as _re


def _inline(s):
    import html as _h
    s = _h.escape(s)
    s = _re.sub(r"\*\*(.+?)\*\*", r"<strong>\1</strong>", s)
    s = _re.sub(r"`(.+?)`", r"<code>\1</code>", s)
    return s


def md_to_html(text):
    out, in_tbl = [], False
    for ln in text.split("\n"):
        s = ln.rstrip()
        if s.startswith("|") and s.endswith("|"):
            cells = [c.strip() for c in s.strip("|").split("|")]
            if set("".join(cells)) <= set("-: "):
                continue
            tag = "td" if in_tbl else "th"
            if not in_tbl:
                out.append("<table>"); in_tbl = True
            out.append("<tr>" + "".join(f"<{tag}>{_inline(c)}</{tag}>" for c in cells) + "</tr>")
            continue
        if in_tbl:
            out.append("</table>"); in_tbl = False
        if s.startswith("### "):
            out.append(f"<h3>{_inline(s[4:])}</h3>")
        elif s.startswith("## "):
            out.append(f"<h2>{_inline(s[3:])}</h2>")
        elif s.startswith("# "):
            out.append(f"<h1>{_inline(s[2:])}</h1>")
        elif s.startswith("> "):
            out.append(f"<blockquote>{_inline(s[2:])}</blockquote>")
        elif s.startswith("- "):
            out.append(f"<li>{_inline(s[2:])}</li>")
        elif s.strip() == "---":
            out.append("<hr>")
        elif s.strip() == "":
            out.append("")
        else:
            out.append(f"<p>{_inline(s)}</p>")
    if in_tbl:
        out.append("</table>")
    body = "\n".join(out)
    return ("<!doctype html><meta charset='utf-8'><title>Geometry-Colour feasibility report</title>"
            "<style>body{font:14px/1.6 -apple-system,Segoe UI,Roboto,sans-serif;max-width:900px;"
            "margin:2rem auto;padding:0 1rem;color:#222}table{border-collapse:collapse;margin:1rem 0;"
            "font-size:12px;display:block;overflow-x:auto}th,td{border:1px solid #ddd;padding:4px 8px;"
            "text-align:right}th{background:#f4f4f4}td:first-child,th:first-child{text-align:left}"
            "h1,h2{border-bottom:1px solid #eee;padding-bottom:.2em}blockquote{border-left:3px solid "
            "#e0574c;margin:1em 0;padding:.2em 1em;background:#fff6f5}</style>" + body)


html_path = os.path.join(RESULTS_DIR, "experiment_report.html")
with open(html_path, "w") as f:
    f.write(md_to_html(REPORT_MD))
print("html report ->", html_path)

print("\n" + "=" * 78)
print("SAVED OUTPUTS")
print("=" * 78)
for label, d in (("report", RESULTS_DIR), ("checkpoints", CKPT_DIR), ("history", HIST_DIR),
                 ("tables", TAB_DIR), ("figures", FIG_DIR)):
    files = sorted(f for f in os.listdir(d) if os.path.isfile(os.path.join(d, f)))
    print(f"\n{label}  ({d})")
    for f in files:
        print(f"   {f:52s} {os.path.getsize(os.path.join(d, f))/1024:8.1f} KB")

## 24 · Next-step recommendation

§12 of the generated report names the next experiment based on what actually happened. The cell
below reprints it alongside the levers that matter most.

In [ ]:
print("=" * 78)
print("NEXT STEP — chosen by this run's outcome (§12 of the report)")
print("=" * 78)
if "REPORT_MD" in dir():
    seg = REPORT_MD.split("## 12. Recommended next experiment")
    print(seg[-1].split("\n---")[0].strip() if len(seg) > 1 else "(report not generated)")

print("\n" + "=" * 78)
print("CHEAPEST LEVERS, IN ORDER")
print("=" * 78)
print('''
 1. Run §9/§10 on the other two categories (change CATEGORY, run to §10, ~1 min each). The
    coherence ratio R and its random-partition null tell you whether the premise even holds
    BEFORE spending 25 minutes on training. Pick the category with the largest R-to-null gap.
 2. More validation objects (NUM_VAL_SHAPES 40 -> 60) and EVAL_REPEATS 4 -> 8. The bootstrap
    intervals shrink as sqrt(n); most of this study's comparisons are small effects.
 3. Three seeds (SEED = 0,1,2). Single-seed intervals cover variation across objects, not across
    training runs.
 4. Finer part units: replace the semantic partition with the FPS-Voronoi cells already used by
    region_dE. If purely geometric cells work as well as semantic parts, the useful structure is
    locality, not semantics -- a genuinely different conclusion.
 5. Soft masking (attention weighted by same-part membership) instead of a hard cut, if and only
    if the boundary metric showed degradation.

 Deliberately NOT next: full 6D joint diffusion, a reverse sampler, or predicted-part segmentation
 layered on top of an unresolved oracle result. Each adds an error source; this notebook exists to
 keep them separate.
''')
print("results directory:", RESULTS_DIR)